# <center>Codex Loop Engineering：循环工程实践</center>

&emsp;&emsp;在前面的 Codex 课程里，我们已经用过 Codex App、Goal、Automations、Skills、worktree、多会话协同和真实项目开发流程。到这一步，AI 编程不再只是“写一句更好的提示词”，而是开始面对真实工程里的连续推进任务。

&emsp;&emsp;本课聚焦的，就是把这种连续推进组织成一套<font color=red>可控工程循环</font>。单次 Prompt 能让本轮输入更清楚，Context 能让模型看到合适材料，Harness 能把一次运行接上工具、权限、验证和人工门；Loop 则进一步处理跨轮触发、状态记录、检查反馈、修正、升级给人和停止。我们关心的不是让 agent 一直运行，而是让每一轮都有证据、有边界、有复核、有停止依据。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/embedded-cell-003-ec459799.png" width=80%></div>


&emsp;&emsp;全课分四章走完这条路线。第一章讲 Prompt → Context → Harness → Loop 的责任演进，先把四层边界立起来；第二章观察每日 AI 资讯任务怎样借助 Automation、Skill、检测脚本、state、verifier 和 Goal 逐步补齐闭环；第三章手动构建loop engineering，不使用内置 Goal 时怎样通过 Git baseline、worktree、迁移地图、检查脚本、状态文件、和 AGENTS.md 手动搭出类似goal的循环任务；第四章再把前三章收束成可触发、可观察、可停止、可沉淀的 loop 设计清单。

## <center>第一章：从 Prompt 到 Loop：理解循环工程</center>

&emsp;&emsp;真实任务很少在一次说明、一次生成、一次验收里结束。我们面对的不是“让 LLM（Large Language Model，大语言模型）回答一句话”这么简单，而是要让它在材料、工具、检查和人工判断之间持续协作，最后得到一个能落地、能复盘、能停止的结果。

&emsp;&emsp;本章讲的 Prompt Engineering、Context Engineering、Harness Engineering 和 Loop Engineering，不是四个突然冒出来的新名词。更通俗地看，它们是 AI 智力越来越强、上下文窗口越来越大之后，一步步给它补“做事条件”的过程。早期我们最关心的是把话说清楚，所以有 Prompt Engineering；后来模型能读更多材料，我们就开始关心应该给它看什么，所以有 Context Engineering；再后来模型能调工具、改文件、访问外部系统，我们就得关心它能在什么边界里做事，所以有 Harness Engineering；等任务变成每天跑、反复修、需要验收和停止时，就进入 Loop Engineering。它们不是前一个淘汰后一个，而是叠在一起用：做 Loop 时，仍然要写好 Prompt、准备好 Context、搭好 Harness。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/embedded-cell-007-65b2e9bd.png" width=80%></div>


&emsp;&emsp;这张图先把 LLM 与 Agent 系统的关系放回一条<font color=red>演进线</font>。Agent 系统指围绕模型组织上下文、工具、状态、验证和控制流程的系统形态；在本课采用的分析框架里，这条线从输入工程开始，走到上下文装配，再走到运行外壳，最后进入跨轮控制。接下来我们逐层拆开看，每一层到底多承担了哪一部分工程责任。

### 1.1 Prompt Engineering

&emsp;&emsp;Prompt engineering（提示词工程）就是设计和优化你给 AI 的指令，让它更准确地理解你的需求，并输出更有用的结果。你可以把 prompt 想成“跟 AI 沟通的说明书”：你说得越清楚，AI 越容易给出符合预期的回答。

&emsp;&emsp;它不只是“问问题”，还包括告诉 AI：你想要什么角色、什么背景、什么格式、什么语气、回答多详细、要不要举例、有什么限制条件。比如你直接问“写一份报告”可能结果很泛；但你说“请以市场分析师的身份，写一份面向高管的 800 字新能源车市场报告，包含趋势、风险和建议”，结果通常会更精准。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/embedded-cell-012-aaf59bc5.png" width=80%></div>


&emsp;&emsp;2020-2022 年可以看作 Prompt Engineering 的实践成形期。2020 年 GPT-3 的 few-shot prompting 让开发者看到“用文本交互指定任务和示例”的威力；2021 年 prompt-based learning survey 把这类做法系统整理；2022 年 Chain-of-Thought prompting 又强化了用提示词组织推理过程的实践。它在这一阶段逐渐成为开发者影响模型输出的工程入口。

&emsp;&emsp;<font color=red>prompt engineering</font>的价值非常明确：它能显著改变当前这一轮输出。我们把“本轮要完成什么”“不能违反哪些约束”“需要参考哪些背景”“结果按什么格式给出”“结束前怎样验收”写清楚，模型就更容易生成可用结果。Prompt 始终是 LLM 接收任务的入口，后面的 Context、Harness 和 Loop 都建立在清晰输入之上。

&emsp;&emsp;Prompt engineering 的局限性在于：它主要是在优化“怎么问”。
比如我们把“帮我写咖啡店文案”改成“你是一名小红书博主，请按标题、正文、标签输出”，确实会好很多。但如果 AI 不知道这家店的真实位置、菜单、价格、用户评价、图片风格，光靠 prompt 写得再漂亮，也容易生成空泛内容，甚至编造信息。

&emsp;&emsp;第二个局限是：prompt 往往是静态的。
它适合一次性任务，但面对复杂任务就不够了。比如做一份真实探店内容，AI 可能需要读取店铺资料、查询地图、参考历史笔记、理解用户偏好、筛选图片、按平台风格输出。你不可能把所有信息都手动塞进一个 prompt 里；塞太多还会让上下文变臃肿，模型反而容易抓不住重点。Anthropic 也提到，context 是有限资源，越长不一定越好，关键是管理好进入模型的有效信息。

### 1.2 Context Engineering

&emsp;&emsp;Context engineering （上下文工程）就是为了解决这个问题：它不只优化“怎么问”，而是优化“AI 在回答前能看到什么”。
Anthropic 把 context engineering 视为 prompt engineering 的自然延伸：不只是写系统提示词，而是管理推理时进入模型的全部信息，包括系统指令、工具、外部数据、历史消息等。 LangChain 的说法更工程化：context engineering 是构建动态系统，把正确的信息、工具和格式提供给 LLM，让它更有可能完成任务

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/embedded-cell-019-66bf82eb.png" width=80%></div>


&emsp;&emsp;到 2025 年，Context Engineering 这个说法开始成为工业级 LLM 应用里更清楚的表达。尤其是 2025-06-25，Andrej Karpathy 在 X 上的帖子让 `context engineering` 这个说法被更广泛讨论和强化，用来强调关键工作是把合适的信息填进上下文窗口。这个时间锚点说明，主线已经从组织提问，推进到组织材料。

> <font size=2><b>【来源说明】<font color=red>Context Engineering 时间锚点</font></b></font>
> <font size=2>2025-06-25，发布载体：X 发文；原始发文：Andrej Karpathy 讨论用 `context engineering` 描述工业级 LLM 应用中的上下文装配工作；完整地址：https://x.com/karpathy/status/1937902205765607626</font>

&emsp;&emsp;Context Engineering 处理的是 LLM 下一步该看什么。模型本身根据当前可见上下文生成内容；天气预报、家庭日历、景点营业时间、地图耗时，以及从外部状态记录提炼出的本轮可见状态摘要，都需要由外部系统先收集、筛选、压缩整理，再送进模型的可见范围。放到 AI 新闻日报里，就是先找新闻、读原文、提取标题 / 来源 / 时间 / 图片 / 链接，再去重筛选，最后整理成 `verified_news_context` 给模型生成日报。RAG、embedding、vector database、chunking、rerank、context compression、memory、citation 等术语，都可以理解为这条材料链路里的具体技术手段；初学时先抓住主线：把材料准备对。

&emsp;&emsp;但 Context Engineering 也有自己的边界。它主要关注的是“信息供给”，也就是把什么内容放进模型上下文里；可是当模型开始调用工具、读取文件、访问网页、执行代码、连接数据库时，问题就不只是“给模型看什么”了。我们还需要考虑：模型能用哪些工具、什么时候用工具、工具返回结果怎么处理、失败后怎么兜底、敏感操作怎么限制、最终结果怎么验证。


&emsp;&emsp;Context Engineering 能提升模型“理解材料和生成内容”的能力，但还不足以支撑一个真正可以执行任务的 Agent。因为真实任务不仅需要上下文，还需要工具、状态、权限、验证、日志和错误处理等外层工程机制。也正是在这个地方，我们需要继续往外走一步，引入 Harness Engineering。


### 1.3 Harness Engineering

&emsp;&emsp;Harness Engineering（驾驭工程） 就是管理 LLM 如何在既定环境里，安全、可控、可验证地执行任务。更直白地说，Context 决定模型看什么，Harness 决定模型能在什么边界里做什么。当 LLM 开始调用工具、读取文件、访问网页、执行代码、连接数据库时，我们就需要为它设计一套外层运行环境：限制它能做什么、决定它能用哪些工具、记录它做了什么、处理失败情况，并通过验证机制判断结果是否可靠。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/embedded-cell-027-596a99a2.png" width=80%></div>


&emsp;&emsp;2026-04-19，Addy Osmani 发布 `Agent Harness Engineering`，把 Context 继续外扩到工具、权限、状态、验证、恢复路径和人工门。Harness Engineering 处理的是 agent 的运行外壳。单轮输入让模型知道任务，上下文让模型看到材料；要让任务真正进入执行，还需要把 LLM、提示词、上下文策略、工具、权限、状态、错误处理、验证信号和人工门组合起来，形成一个可执行、可约束、可验证的运行单元。Addy 在文中的口径可以概括为：Agent = Model + Harness，也就是模型外面还要有 prompts、tools、context policies、hooks、sandboxes、subagents、feedback loops 和 recovery paths 共同构成运行外壳。

> <font size=2><b>【名词解释】<font color=red>Agent Harness Engineering</font>（Agent Harness Engineering，Agent 运行外壳工程）</b></font>
> <font size=2>2026-04-19，发布载体：Addy Osmani 个人博客文章；文章：`Agent Harness Engineering`；完整地址：https://addyosmani.com/blog/agent-harness-engineering/</font>
> <font size=2>含义：围绕模型搭建 prompts、tools、context policies、hooks、sandboxes、subagents、feedback loops 和 recovery paths 的工程方法。</font>

&emsp;&emsp;模型负责根据输入生成下一步建议或内容，harness engineering负责让一次运行进入可观察、可限制、可验证的环境。因此Harness Engineering 建立在 Prompt 和 Context 之上：它继续使用目标、约束和材料装配，再额外加入外部执行接口、权限、验证和人工确认。

&emsp;&emsp;Harness Engineering 已经为 Agent 搭好了一个相对完整的执行环境：模型可以在权限边界内调用工具、读取文件、访问网页、执行代码，并通过日志、状态、验证规则和错误处理来提高可靠性。

&emsp;&emsp;但当任务从“一次性执行”变成“持续性工作”时，问题又会进一步变化。我们不仅要关心 Agent 能不能安全执行工具，还要关心它什么时候被触发、每一轮目标是什么、如何判断任务真的完成、执行状态如何保存、下次任务如何接上，以及是否需要另一个 verifier 或 subagent 来检查结果。




### 1.4 Loop Engineering

&emsp;&emsp;Loop Engineering（循环工程） 是在 Harness 之上设计一个持续运行的工作循环。Harness 让 Agent 能可靠地做事；Loop 则让 Agent 能在重复任务、长期目标和外部验证机制中稳定推进。可以理解为：不再由人一轮一轮地提示 Agent，而是设计一个循环系统，让 Agent 在规则、状态和验证机制里自己持续推进任务。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/embedded-cell-035-ce12aa6a.png" width=80%></div>


&emsp;&emsp;它不是代码里的 for 循环，而是一种 Agent 工作方式：先定义目标、触发条件、可用工具、状态记录、检查规则和停止条件，然后让 Agent 每一轮都按这个机制执行、检查、修复，直到任务完成。这里可以把 loop 拆成两层：内循环关心“当前这一个任务是否完成”，每一轮都根据检查结果决定继续修、交给人还是停止；外循环关心“整个任务要不要常态化反复运行”，比如每天、每周，或者遇到某个外部事件时重新启动一次任务。

&emsp;&emsp;对应到产品形态里，内循环已经有现成入口：Claude Code 和 Codex 都提供 `/goal` / Goal mode，用完成条件推动当前任务跨多轮执行，直到达到停止条件或需要人工介入。外循环更接近调度层：Claude Code 里可以用 `/loop` 在打开的会话中重复运行 prompt；Codex 里对应 Automations，也就是本课按当前中文界面写作“已安排”的入口，负责把稳定任务按时间或规则再次触发。简单说，`/goal` 解决“这一轮要不要继续”，`/loop` / “已安排”解决“这个任务以后还要不要定期再跑”。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/embedded-cell-036-53bf37b1.png" width=80%></div>


&emsp;&emsp;2026-06-07，Addy Osmani 发布 `Loop Engineering`，把 automations、worktrees、skills、plugins/connectors、subagents 和外部 memory 都列为 loop 的关键组成。把 Harness 继续外扩到跨轮控制，Loop Engineering 处理的是 harness 之上的迭代控制系统。它关注的不是某一次输入，也不是某一批上下文，而是多轮任务怎样发现、分配、执行、检查、记录、继续、修正、升级给人或停止。Addy 在文中给出的核心口径是：loop sits one floor above the harness，它用外层系统替代“人反复 prompt agent”，让系统围绕目标发现任务、分配任务、检查结果、写下完成状态，再决定下一步。

> <font size=2><b>【名词解释】<font color=red>Loop Engineering</font>（Loop Engineering，循环工程）</b></font>
> <font size=2>2026-06-07，发布载体：Addy Osmani 个人博客文章；文章：`Loop Engineering`；完整地址：https://addyosmani.com/blog/loop-engineering/</font>
> <font size=2>含义：harness 之上的外层迭代控制系统，让 AI agent 围绕目标持续发现任务、执行、检查、记录状态并决定继续或停止。</font>

&emsp;&emsp;把这套外层控制落到 Codex 场景时，Addy 文中提到的一组外部部件可以作为实现视角：<font color=red>Automations</font>、Worktrees、Skills、Plugins and connectors、Sub-agents，以及 memory/state。Automations 对应本课里说的自动化 / 已安排入口，负责让任务按时间或规则启动；Worktrees 让并行尝试互不覆盖；Skills 把可复用流程和项目知识写下来；Plugins and connectors 把日历、工单、文档系统、消息系统等外部工具接进 loop；Sub-agents 分别承担执行、审查或验收；memory/state 把完成状态、失败原因、下一步和经验沉淀到单次对话之外。这些是扩展和落地视角，不是理解 Loop Engineering 主定义的前置条件。

<div align=center><font size=2 color=#999999>Loop Engineering 的三类触发方式</font></div>

<div align="center">

<table width="80%">

<thead>

<tr><th>触发方式</th><th>启动条件</th><th>例子</th></tr>

</thead>

<tbody>

<tr><td>人工触发</td><td>人主动给出目标、反馈或确认。</td><td>用户点击“重新生成报告”，Agent 读取最新数据、生成新版内容，并根据用户反馈继续修改，直到用户确认可用。</td></tr>

<tr><td>定时触发</td><td>到固定时间自动启动任务。</td><td>每周一早上 9 点，Agent 自动汇总上周会议纪要、待办事项和项目进展，生成一份周报草稿。</td></tr>

<tr><td>条件触发</td><td>某个外部事件、状态变化或检查结果满足条件。</td><td>当库存低于安全阈值时，Agent 自动检查近期销量、供应周期和库存记录，生成补货建议并提交给负责人确认。</td></tr>

</tbody>

</table>

</div>


&emsp;&emsp;普通脚本按固定步骤执行，适合路径稳定、输入输出明确的任务。loop 会根据状态、要求、检查者判断和反馈结果决定下一步，适合多轮探索、修复、归因和收敛。它不保证自动成功，但能让每一轮留下证据，让我们知道应该继续、修正、交给人、停止还是沉淀。因此在本课框架里，Loop Engineering 建立在 Prompt、Context 和 Harness 之上：它仍然需要清晰输入、材料装配和运行外壳，只是在更外层加入检查者、外部控制状态、完成判断、反馈入口、经验沉淀和停止条件。

> <b>提示</b>: loop 的核心不是让系统一直运行，而是每一轮都有检查者、状态、证据、反馈入口、接手对象、经验沉淀和停止判断。

### 1.5 从 Prompt Engineering 到 Loop Engineering：以减脂健身为例

#### Prompt Engineering：把需求说清楚

&emsp;&emsp;我们先用减脂健身这个更常见、非代码的任务来串起四层工程责任。最开始，任务只是“帮我做一个 30 天减脂计划”，这时重点还在把目标、身份、限制条件和输出格式说清楚。

> 💬 输入给 Codex 的提示词

```text
你是一名健身教练。

请帮我制定一个 30 天减脂计划。

我的目标是减掉 3 公斤，平时工作比较忙，每周最多运动 4 次，每次不超过 45 分钟。

要求：
1. 包含训练安排
2. 包含饮食建议
3. 不要太极端
4. 适合普通上班族执行
5. 用表格输出
```

&emsp;&emsp;这就是 Prompt Engineering：把目标、身份、限制条件、输出格式说清楚。它能让模型生成一个看起来不错的通用计划，但模型并不知道真实身高体重、作息、饮食习惯、运动基础、伤病情况、可用器械、通勤时间，也不知道每天实际有没有完成，所以这一层解决的是“怎么说清楚”，还没有解决“模型依据什么个人材料做判断”。

#### Context Engineering：把个人材料准备好

&emsp;&emsp;进入 Context Engineering 之后，我们不再只优化提问方式，而是先把个人材料整理成模型可以使用的上下文。减脂计划真正要贴近现实，至少要看到个人信息、生活节奏、饮食习惯和目标约束。

```text
个人信息：
- 身高、体重、年龄
- 当前体脂或腰围
- 运动基础
- 是否有膝盖、腰、肩等伤病

生活信息：
- 每天几点起床、几点下班
- 每周哪几天有空
- 是否能去健身房
- 是否能自己做饭

饮食信息：
- 平时早餐、午餐、晚餐吃什么
- 是否经常喝奶茶、吃夜宵
- 是否有忌口
- 每天大概摄入多少热量

目标信息：
- 想减脂还是塑形
- 目标周期
- 可接受的运动强度
```

&emsp;&emsp;有了这份上下文之后，再让模型制定计划，任务就从“泛泛给建议”变成“基于个人情况安排可执行方案”。

> 💬 输入给 Codex 的提示词

```text
请基于下面的个人信息，制定一个 30 天减脂计划。
不要给通用建议，要结合我的作息、饮食习惯、运动基础和可用时间安排。

个人上下文：
{{user_fitness_context}}
```

&emsp;&emsp;这时生成的计划会更贴近现实。Prompt Engineering 解决“怎么把需求说清楚”，Context Engineering 解决“模型制定计划前应该了解哪些个人信息”。不过计划再准确，模型仍然没有真正帮我们执行：它不会自动安排日历，不会记录每天是否完成，也不会根据体重变化调整后续计划。

#### Harness Engineering：搭好执行环境

&emsp;&emsp;Harness Engineering 会把模型接入真实工具，让它在受控环境里帮我们执行任务。减脂助手不只是写一张表，而是可以连接日历、提醒、饮食记录、体重记录、运动记录和计算工具，同时受到明确的安全边界约束。

```text
工具：
- 日历工具：把训练安排写入日历
- 饮食记录工具：记录每天吃了什么
- 体重记录工具：记录每天体重和腰围
- 运动记录工具：记录跑步、力量训练、步数
- 提醒工具：提醒训练、喝水、早睡
- 计算工具：估算热量缺口和蛋白质摄入

权限边界：
- 不能给极端节食建议
- 不能替代医生诊断
- 不能安排超过用户承受能力的训练
- 如果出现疼痛、头晕、异常疲劳，要建议停止并寻求专业帮助
```

&emsp;&emsp;接入这些工具之后，模型不只是“生成计划”，而是在安全边界内把计划推进到真实执行。

```text
生成训练计划
↓
写入日历
↓
提醒用户训练
↓
记录饮食和体重
↓
计算执行情况
↓
根据数据给出调整建议
```

&emsp;&emsp;在这里，Context Engineering 让模型更懂个人情况；Harness Engineering 让模型能借助工具，把一次计划安排、提醒、记录和计算串起来。

#### Loop Engineering：形成持续反馈循环

&emsp;&emsp;减脂不是一次性任务，不是生成一张表就结束。真正难的是持续执行和动态调整，所以 Loop Engineering 会把它设计成一个跨天、跨周、可检查、可修正的反馈循环。

```text
每天早上：
记录体重和睡眠

每天晚上：
记录饮食和运动完成情况

每周一次：
汇总体重、腰围、训练完成率、饮食情况

然后判断：
- 如果体重下降正常，继续当前计划
- 如果连续两周没有变化，微调饮食或运动
- 如果疲劳过高，降低训练强度
- 如果执行率太低，重新安排更现实的计划

Verifier：
检查计划是否过度激进
检查热量缺口是否合理
检查训练安排是否有恢复日
检查用户是否真的完成了本周任务
```

&emsp;&emsp;这时候，系统不只是制定计划，而是在持续运行：记录执行、检查结果、发现问题、调整方案，再进入下一周。

```text
制定计划
↓
执行记录
↓
检查结果
↓
发现问题
↓
调整计划
↓
继续下一周
```

&emsp;&emsp;因此，Harness Engineering 让 Agent 能安全执行一次计划；Loop Engineering 则让 Agent 能根据反馈持续调整计划，直到目标达成或需要人工重新判断。

#### 最后总结

```text
Prompt Engineering：
写清楚“我要一个 30 天减脂计划”。

Context Engineering：
提供身高体重、作息、饮食、运动基础，让计划更个性化。

Harness Engineering：
接入日历、提醒、饮食记录、体重记录等工具，让计划可以被执行和追踪。

Loop Engineering：
每天记录、每周复盘、根据结果调整计划，让减脂变成一个持续反馈循环。

核心一句话：
Prompt 解决“怎么说”；
Context 解决“了解我”；
Harness 解决“帮我做”；
Loop 解决“持续改”。
```

### 1.6 本章回顾

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/embedded-cell-070-69cd11e7.png" width=80%></div>


&emsp;&emsp;本章按时间线回顾了 LLM 到 Agent 系统的工程演进：2020-2022 年，Prompt Engineering 让当前这一轮生成被目标和约束条件化；2022-2025 年，ReAct、Toolformer 和工具调用实践逐步把模型带向外部材料、工具、状态和编排；2025 年，Context Engineering 把重点收束到本轮材料装配；2026-04-19，Agent Harness Engineering 把一次运行接入工具、权限、验证、恢复和人工门；2026-06-07 以后，Loop Engineering 把关注点继续推到人工触发、定时触发、条件触发、跨轮检查、修正、记录、沉淀和停止判断。

## <center>第二章：Codex loop engineering实践</center>

&emsp;&emsp;第一章里，我们已经把 Prompt、Context、Harness 和 Loop 四层责任区分清楚了。第二章我们选择看一个每日 AI 资讯任务怎样从手动 prompt，逐步补齐 Automation、Skill、memory、检测脚本、state、verifier 和 Goal，变成可以被检查、记录和停止的 loop engineering。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/embedded-cell-074-9cf825db.png" width=80%></div>


### 2.1 手动任务升级为自动化任务

&emsp;&emsp;在之前《OpenAI Codex 快速入门》的课程中，我们已经尝试过让 Codex 检索过去 24 小时内 AI 圈的重要新闻，确认原文链接、发布时间、来源和图片，再生成一份本地 HTML 日报，并自动打开浏览器展示。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch02-daily-ai-original-01-3b5c11e5.png" width=80%></div>

&emsp;&emsp;这个案例已经包含了一部分 loop engineering 的思想：把原本每天都需要人工重复执行的任务，交给 Codex <font color=red>自动完成</font>。

&emsp;&emsp;不过当时这个任务本质上还是一次性的。也就是说，我们每次都需要手动告诉 Codex：

```text
去检索今天的 AI 新闻
确认来源和发布时间
生成 HTML 日报
保存到本地
打开浏览器
```

&emsp;&emsp;这当然已经比纯手工操作方便很多，但它还没有真正形成一个<font color=red>长期运行的循环系统</font>。

&emsp;&emsp;如果想进一步减少人工介入，就需要把它改造成一个可以每天自动触发的任务。这里我们可以直接使用 Codex 的自动化功能，让它每天固定时间运行。这里的“每天固定时间运行”有本地项目级自动化前提：运行 Codex App 的机器需要开机，Codex 保持运行，并且项目目录仍然可访问；否则到点不会执行。

&emsp;&emsp;因为我们之前已经做过一次类似案例，所以这次不再一步步创建，而是直接用一个 prompt，把「自动化任务」和「HTML 风格 Skill」一次性让 Codex 建好。我们把下面这些内容发给 Codex：

> <b>提示</b>: 这段长 prompt 只用于首次搭建。阅读时抓三层：建 Skill、建 Automation、写 Automation 内部执行 Prompt。

> 💬 输入给 Codex 的提示词

```text
我想创建一个“每日 AI 资讯自动化任务”，并同时创建一个用于美化 HTML 日报的 Codex Skill。

目标是：让 Codex 每天自动检索过去 24 小时内 AI 圈的重要新闻，整理成一份图文并茂的中文 HTML 日报，保存到本地目录，并自动用浏览器打开。同时，HTML 页面需要使用我指定的 Scatterbrain 风格 Skill 来生成，整体像便利贴、软木板、纸张纹理风格的资讯看板。

请你帮我完成以下两件事：

1. 创建一个名为 scatterbrain-html-daily 的 Codex Skill。
2. 创建一个每日 9 点执行的自动化任务，使用这个 Skill 生成每日 AI 资讯 HTML 日报。

⸻

一、创建 Skill：scatterbrain-html-daily

我喜欢这个 HTML 模板的视觉风格：

https://github.com/zarazhangrui/beautiful-html-templates/tree/main/templates/scatterbrain

请帮我把它创建成一个 Codex Skill。

Skill 名称：

scatterbrain-html-daily

Skill 保存路径：

skills/scatterbrain-html-daily/SKILL.md

Skill 用途：

当我要求生成每日资讯、AI 日报、新闻简报、图文 HTML 日报、本地 HTML 报告时，使用这个 Skill 的视觉风格。

请完成以下工作：

1. 读取该模板目录下的 design.md、template.html、template.json。
2. 总结 Scatterbrain 的核心视觉规范。
3. 创建一个可复用的 Skill，而不是只复制一次 HTML。
4. SKILL.md 要说明触发场景、页面结构、颜色、字体、新闻卡片布局、响应式规则、链接规则。
5. 在 references/ 中保存该风格的设计规范摘要。
6. 在 templates/ 或 assets/ 中保存可复用的 HTML 模板骨架。
7. 这个 Skill 只负责 HTML 视觉呈现，不负责联网搜索新闻。
8. 后续我的每日定时任务生成 HTML 日报时，需要使用这个 Skill 的视觉规范。

Scatterbrain 风格要求：

* 整体像软木板、纸张、便利贴组成的新闻看板。
* 背景可以有纸张纹理、软木板感或浅色手作感。
* 新闻卡片像便利贴或纸片。
* 卡片可以有轻微旋转、错落排列、柔和阴影。
* 可以使用图钉、胶带、标签等轻量装饰。
* 色彩可以使用浅黄、浅蓝、浅绿、浅粉等便利贴颜色。
* 字体要有轻微手写感或轻松阅读感，但不能影响中文可读性。
* 页面整体要适合阅读、截图和分享。
* 装饰不能喧宾夺主，新闻内容必须清晰。
* 所有新闻卡片必须支持点击跳转到原文。
* 标题、图片、「查看原文」按钮都应该能跳转到原文。
* 移动端需要单列展示，桌面端可以多列展示。
* 图片不能拉伸变形，统一使用合适比例展示。

请创建完成后，告诉我 Skill 的名称、保存位置、触发场景，以及以后如何在任务里指定使用它。

⸻

二、创建每日 AI 资讯自动化任务

请创建一个 Codex 自动化任务。

任务名称：

每日AI资讯

执行频率：

每天上午 9 点执行。

运行环境：

本地。

运行目录：

当前项目根目录（daily-AI-news-loop）

请设置自动化任务在当前项目根目录运行。如果当前目录不是 daily-AI-news-loop，请停止并说明，不要在其他目录生成。

如果当前环境无法写入该目录，请停止任务，并告诉我需要授予本机文件访问权限，不要只返回文字版日报。

⸻

三、自动化任务 Prompt

请把下面内容作为自动化任务的执行 Prompt：

请每天检索过去 24 小时内 AI 圈的重要新闻，并生成一份图文并茂的中文 HTML 日报。

本任务只在本机目录写入文件，不要上传 GitHub，不要执行 git push，不要提交 commit。

任务开始前请先检查：

1. 当前环境是否可以写入当前项目根目录
2. 如果当前目录不是 daily-AI-news-loop，请停止并说明，不要在其他目录生成。
3. 如果无法写入该目录，请先停止任务并告诉我需要授予本机文件访问权限，不要只返回文字版日报。
4. 当前是否可以在本机打开浏览器。
5. 如果无法自动打开浏览器，请在完成后明确告诉我 HTML 文件的项目内相对路径。

⸻

四、检索策略

* 优先使用 Codex 可用的联网搜索和网页浏览能力完成检索、来源确认、原文链接提取和图片链接提取。
* 只有在普通搜索或网页浏览无法获取页面内容、图片链接、发布时间，或者必须使用已登录浏览器状态时，才使用 Chrome 插件或界面操作能力。
* 不要只根据搜索结果摘要写新闻，重要新闻必须进入原文页面确认。
* 不要凭记忆写，必须基于当天搜索结果。
* 不要堆砌小新闻，优先筛选真正重要的消息。

⸻

五、保存与展示要求

1. 每天先创建当日目录：
   runs/YYYY-MM-DD
2. 将 HTML 文件保存到当日目录：
   runs/YYYY-MM-DD/AI日报-YYYY-MM-DD.html
3. 文件名格式：
   AI日报-YYYY-MM-DD.html
4. 生成完成后，用默认浏览器打开该 HTML 文件。
5. 如果需要用 Chrome 打开本地 HTML，请使用 Chrome 插件。
6. 如果 Chrome 无法访问 file:// 本地文件，请提示我开启 Codex Chrome 扩展的 Allow access to file URLs 权限。

⸻

六、链接要求

* 每条新闻必须有真实原文链接。
* 新闻卡片整体可以点击跳转。
* 标题、图片、「查看原文」按钮都能跳转到原文。
* 所有链接使用 target="_blank" 打开。
* 不要使用搜索结果页链接。
* 没有可靠原文链接的新闻不要收录。

⸻

七、图片要求

* 优先引用原文页面中的代表性图片。
* 图片直接使用原文图片 URL，不下载到本地。
* 图片点击后跳转到原文。
* 如果没有可靠原文图片，就不要强行配图。
* 不要使用无来源图片、无关配图、AI 生成配图。

⸻

八、事实校验要求

* 每条新闻必须确认发布时间、来源媒体或来源机构、原文 URL。
* 涉及模型发布、价格、API 更新、公司公告时，优先使用官方博客、官方文档、GitHub Release、论文页面或公司公告。
* 如果只有二手转载，请标注「二手来源」。
* 不确定的信息不要写成确定事实。
* “新闻事实”和“为什么重要”分开写，不要把主观判断伪装成事实。
* 如果无法打开原文确认发布时间和来源，则不要收录该新闻。

⸻

九、内容要求

新闻内容需要覆盖以下方向：

1. 模型动态
2. 产品工具
3. Agent 与编程工具
4. 开源项目
5. 研究论文
6. 行业商业

每条新闻包含：

* 标题
* 摘要
* 为什么重要
* 发布时间
* 来源
* 原文链接
* 图片来源

⸻

十、页面结构

HTML 页面结构如下：

1. 每日 AI 资讯
2. 日期与更新时间
3. 今日最重要的 5 条
4. 模型动态
5. 产品工具
6. Agent 与编程工具
7. 开源项目
8. 研究论文
9. 行业商业
10. 今日最值得关注的一件事

⸻

十一、HTML 设计要求

生成 HTML 时必须使用 scatterbrain-html-daily Skill 的视觉规范。

具体要求：

* 单文件 HTML，CSS 内嵌。
* 整体采用 Scatterbrain 风格：便利贴、软木板或纸张纹理、图钉、胶带、轻微旋转、柔和阴影、彩色新闻卡片。
* 页面要适合直接阅读、截图和分享。
* 新闻卡片可以有轻微错落感，但不能影响可读性。
* 桌面端多列布局，移动端单列布局。
* 图片比例统一，使用 object-fit: cover，不能拉伸变形。
* 标题、摘要、来源、时间层级清晰。
* 不要使用无关装饰影响新闻阅读。
* 不要使用花哨渐变、夸张阴影、紫色 AI 风格。
* 所有 CSS 写在 HTML 内部，不依赖外部 CSS 文件。
* 页面必须能作为本地静态 HTML 直接打开。

⸻

十二、任务完成后的输出

每次任务完成后，请在 Codex 会话中输出简短总结，包括：

1. 本次生成的 HTML 文件路径。
2. 是否成功打开浏览器。
3. 收录了多少条新闻。
4. 是否有新闻因为无法确认来源、发布时间或原文链接而被排除。
5. 是否成功使用了 scatterbrain-html-daily Skill。
6. 如果浏览器无法打开，请明确告诉我 HTML 文件的项目内相对路径。

⸻

十三、注意事项

* 不要创建 GitHub 仓库。
* 不要上传任何文件。
* 不要执行 git push。
* 不要提交 commit。
* 不要把日报只返回在聊天窗口里，必须写入本地 HTML 文件。
* 不要生成 Markdown 日报，必须生成 HTML。
* 不要使用虚构新闻。
* 不要使用无法点击的假链接。
* 不要使用无来源图片。
* 不要使用搜索结果页作为新闻原文。

请现在完成：

1. 创建 scatterbrain-html-daily Skill。
2. 创建每日 9 点执行的 每日AI资讯 自动化任务。
3. 设置自动化任务使用本地运行环境。
4. 设置自动化任务的运行目录为当前项目根目录（daily-AI-news-loop）
5. 创建完成后，告诉我 Skill 和自动化任务是否创建成功。
```


&emsp;&emsp;发送之后，可以看到 Codex 已经创建了 `scatterbrain-html-daily` Skill，并且每日 AI 资讯自动化任务也已经创建完成。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch02-daily-ai-original-03-5b7f571c.png" width=80%></div>

&emsp;&emsp;接下来，我们可以点击「立即运行」按钮，先手动触发一次自动化任务，看看它能不能正常生成日报。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch02-daily-ai-original-04-850aa14a.png" width=80%></div>

&emsp;&emsp;运行完成后，可以看到 Codex 会自动生成 HTML 日报，并且本次本地运行还留下了 `memory.md` 记录，总结这次运行中的经验。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch02-daily-ai-original-05-24cc174b.png" width=80%></div>

&emsp;&emsp;比如这里它记录了：

```text
下次注意：automation memory.md 已记录本次运行经验和注意事项。
```


&emsp;&emsp;到这里，这个自动化任务已经解决了两个问题：

```text
定时触发：每天 9 点自动运行
跨会话记忆：每次运行后记录经验和注意事项
```

### 2.2 用 Codex 功能搭出 Loop Engineering

&emsp;&emsp;2.1 里，我们已经把每日 AI 资讯做成了“已安排”的自动化任务：它有<font color=red>外循环</font>，每天 9 点会再次触发；但它还不能算完整的 loop engineering，因为它缺少<font color=red>内循环</font>：没有 Goal 模式判断本轮是否完成，也没有检测脚本先验收 HTML。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch02-automation-to-goal-loop-76f0cdeb.png" width=80%></div>


&emsp;&emsp;第一章里我们讲过，Loop Engineering 不是让 agent 一直运行，也不是把 prompt 写得越来越长。它是在 harness 之上的外层迭代控制系统，负责让任务能够被触发、被检查、被记录、被修正，并且知道目标是否真正完成了。

&emsp;&emsp;把这套定义放回每日 AI 日报这个案例里，前面已经完成的是：

```text
Automation：每天 9 点触发
Skill：把 HTML 日报的视觉风格封装成可复用流程
Memory：记录上次运行的经验和注意事项
本地文件系统：保存生成出来的 HTML
```

&emsp;&emsp;这些已经让任务从“手动 prompt”变成了“可自动触发的一次运行”。但是，它还没有回答 loop engineering 最关键的问题：

```text
这轮结果有没有通过检查？
没通过时谁来指出问题？
Codex 要修复还是重跑？
最多修几次？
通过或失败的证据记录在哪里？
什么时候可以停止？
下次运行凭什么不从零开始？
```

&emsp;&emsp;如果这些问题仍然靠人每次打开 HTML 慢慢判断，那它只是一个自动化任务，还不是完整的 loop。

&emsp;&emsp;所以第二步的重点不是再写一个更长的 prompt，而是把第一章里列出的 loop engineering 部件，映射到 Codex 里的具体功能：

<div align="center">
<table width="80%">
<thead>
<tr><th>第一章里的 Loop Engineering 部件</th><th>Codex 里的落点</th><th>在每日 AI 日报里的处理</th></tr>
</thead>
<tbody>
<tr><td>Automations</td><td>每日AI资讯 自动化任务</td><td>每天 9 点自动启动</td></tr>
<tr><td>Worktrees</td><td>Codex worktree</td><td>本案例不需要并行分支；讲到代码迁移、多方案修复时再用</td></tr>
<tr><td>Skills</td><td>scatterbrain-html-daily、daily-ai-report-run-logger</td><td>一个固定 HTML 日报的视觉规范和卡片结构，一个固定当天 run-log.md 的记录格式</td></tr>
<tr><td>Plugins and connectors</td><td>浏览器、GitHub、文档系统等连接器</td><td>本案例不把它作为主角；只有需要浏览器登录态或外部系统时再接入</td></tr>
<tr><td>Sub-agents</td><td>daily-ai-report-verifier</td><td>作为只读审核者，判断新闻质量和页面质量</td></tr>
<tr><td>memory.md</td><td>本次本地运行留下的 automation memory 记录</td><td>只记录跨天有用的经验、排除项和下次注意事项</td></tr>
</tbody>
</table>
</div>


&emsp;&emsp;这样讲，第二章就能和第一章接上：第一章讲 Codex 里这些外层部件可以怎样组成 loop，第二章用每日 AI 日报案例把其中最相关的部件落地。

&emsp;&emsp;这里不需要为了“凑齐名词”强行使用所有 Codex 功能。Worktree 和 plugins/connectors 也是 loop engineering 的组成部分，只是这个日报案例暂时不需要并行分支，也不需要接 GitHub、文档系统或业务系统。它的主线是：Automation 定时触发，Skill 固定输出风格和当日运行记录，memory.md 留下跨天经验，再用 Sub-agent 做独立审核。


> <b>提示</b>: 在这里要注意 Goal 的位置。

&emsp;&emsp;在这个案例里，自动化线程表现为围绕目标持续推进，这是本次实测现象，不应泛化为所有自动化都会隐式进入 Goal 模式。Goal 解决的是“目标是否还在推进、什么时候可以标记完成”的问题，不是“谁来独立审核结果”的问题。所以，想让日报任务更稳，我们就在 Goal 外面再配一个 <font color=red>verifier subagent</font>。

&emsp;&emsp;这个日报任务里，主流程可以用 `gpt-5.5` 跑，因为它要负责检索、判断、生成 HTML、修复错误和推进 Goal。审核 subagent 则可以指定为 `gpt-5.4`，让它和主流程不是同一个模型实例。这是本案例里的成本和独立审核者选择，不代表唯一最佳实践；重点是执行者和审核者分离。这样做不是必需的；不配 subagent，Codex 也可以自己检查。但在 loop engineering 里，我们更希望有一个独立检查者，因为它能减少两类问题：

```text
目标偏移：主流程忙着完成任务，可能慢慢偏离最初的验收标准。
虚假完成：主流程以为自己完成了，但其实链接、来源、时间或页面结构没有过关。
```

&emsp;&emsp;所以这里给 Goal 配一个 verifier subagent。Goal 负责推进，subagent 负责审核，检测脚本负责硬性规则，当天目录里的 `run-log.md` 负责留下每轮证据；只有需要第二天继续注意的问题，才写进 `memory.md`。它们合在一起，才构成这个日报任务的 loop。


&emsp;&emsp;在 Codex 里，这个 loop 可以这样理解：

```text
Automation 到点启动
        ↓
Codex 创建或继续 Goal
        ↓
Codex 执行本轮日报生成
        ↓
检测脚本检查硬性规则
        ↓
gpt-5.4 verifier subagent 审核质量
        ↓
不通过则修复或重跑，最多 2 轮
        ↓
当日 run-log.md 记录轮次、结果和打回原因
        ↓
必要经验写入 memory.md，供第二天参考
        ↓
通过后标记 Goal complete，失败则输出原因
```


&emsp;&emsp;这才是 loop engineering 的思想：不是“把要求塞进一个更大的 prompt”，而是用 Codex 的 Automation、Skill、Sub-agent、当日 `run-log.md` 和 `memory.md` 承接触发、复用、审核、记录和经验沉淀，再用 Goal、检测脚本和停止条件约束这一次日报任务怎么收敛。


&emsp;&emsp;所以接下来不要把所有要求都塞进一个大 prompt。我们把改造拆成四个小 prompt，每一步只补一个能力，让这个 loop 逐层搭出来。

#### 2.2.1 先加硬性检测脚本

&emsp;&emsp;第一步先让 Codex 创建检测脚本。这个脚本相当于第一章里说的 checker / evaluator，负责检查确定性的硬规则。

&emsp;&emsp;可以发给 Codex：

> 💬 输入给 Codex 的提示词

```text
请在当前每日 AI 日报项目里创建硬性检测脚本：

路径：
evals/check_daily_ai_report.py

用途：
检查指定当日运行目录里的 AI日报-YYYY-MM-DD.html 是否满足基础结构和链接规则。

运行方式：
python3 evals/check_daily_ai_report.py \
  --html runs/YYYY-MM-DD/AI日报-YYYY-MM-DD.html \
  --out runs/YYYY-MM-DD/check-result-latest.json

要求：
1. 支持传入 HTML 文件路径，不要只扫描项目根目录。
2. 文件存在，文件名符合 AI日报-YYYY-MM-DD.html。
3. 如果 HTML 位于 runs/YYYY-MM-DD/ 目录下，目录日期要和文件名日期一致。
4. 是单文件 HTML，包含内嵌 CSS，不依赖外部 CSS。
5. 包含「每日 AI 资讯」、日期与更新时间、所有必需栏目。
6. 至少 5 条新闻，至少 5 个唯一可靠原文链接。
7. 每张新闻卡片包含标题、摘要、为什么重要、发布时间、来源、查看原文。
8. 链接不是空链接、假链接或搜索结果页，并使用 target="_blank"。
9. 如果有图片，必须有图片来源，并使用 object-fit: cover。
10. 页面包含 scatterbrain 风格线索：sticky、paper、cork、tape、pin、rotate、box-shadow。

脚本输出 JSON：
{
  "status": "pass | warn | fail",
  "run_date": "YYYY-MM-DD",
  "run_round": "N 或 null",
  "checked_file": "...",
  "output_file": "...",
  "errors": [],
  "warnings": [],
  "summary": "..."
}

创建后请用一个现有 HTML 文件试跑一次，并告诉我结果。
```


&emsp;&emsp;这里的 `pass | warn | fail` 要提前定义清楚，后面 Codex 才知道怎样根据脚本结果决定下一步：

```text
pass：硬性规则全部通过，可以进入 subagent 审核。
warn：没有违反硬性规则，但存在轻微风险或建议项，也可以进入 subagent 审核。
fail：存在硬性问题，不能进入最终通过，需要先修复或重跑。
```

&emsp;&emsp;比如 HTML 文件不存在、栏目缺失、链接为空、用了搜索结果页链接、新闻不足 5 条，这些都应该是 `fail`。如果只是某个 `rel="noopener noreferrer"` 缺失、页面有轻微可访问性提醒，就可以先记为 `warn`，再交给 verifier subagent 做质量判断。

&emsp;&emsp;这个 prompt 不负责“生成日报”，只负责补上 loop 里的硬性检查器。

#### 2.2.2 再加当日运行记录 Skill


&emsp;&emsp;第二步不要做跨日期 state，也不要堆很多轮次 JSON 文件。这里补一个很轻的运行记录 Skill：每天只在当天目录里维护一个 `run-log.md`，记录主 Agent 跑了几轮、每轮结果如何、如果被 verifier 打回是什么原因，以及最终总共跑了几轮。


&emsp;&emsp;可以发给 Codex：

> 💬 输入给 Codex 的提示词

```text
请创建一个轻量 Skill：daily-ai-report-run-logger。

用途：
在每日 AI 日报的当天目录里维护一个 Markdown 运行记录，不创建跨日期 state 文件，也不为每轮 verifier 结果创建一堆 JSON 文件。

Skill 路径：
skills/daily-ai-report-run-logger/SKILL.md

项目目录：
当前项目根目录

每天只写这一个记录文件：
runs/YYYY-MM-DD/run-log.md

run-log.md 至少包含：
# AI 日报运行记录：YYYY-MM-DD

## 今日汇总
- 总轮数：
- 最终状态：pass / failed / waiting_human
- 最终 HTML：
- 最终检测结果：pass / warn / fail
- 最终 verifier 结论：pass / fix_required / rerun_required
- 是否需要写入 memory.md：

## Round N
- 时间：
- 主 Agent 本轮做了什么：
- 检测脚本结果：
- verifier 结论：
- 如果被 verifier 打回，原因是什么：
- 本轮修复或重跑动作：
- 下一步：

要求：
1. 每天只在 runs/YYYY-MM-DD/ 目录下维护 run-log.md。
2. 同一天每跑一轮，就追加或更新一个 Round N 小节。
3. 今日汇总里要能看出总共跑了几轮、最后结果如何。
4. 如果 verifier 打回，必须把打回原因和要求的修复动作写清楚。
5. 不创建根目录跨日期状态文件。
6. 不创建每轮审核 JSON 记录文件；verifier 的结构化输出由主 Agent 摘要写进 run-log.md。
7. 不负责跨天记忆；只有需要第二天继续注意的问题，才由 Codex 另行写入 memory.md。
8. 这个 Skill 只负责运行记录格式，不生成日报，不审核新闻，不替代检测脚本。
```


&emsp;&emsp;这个 Skill 和 `memory.md` 的分工要讲清楚：`run-log.md` 是当天目录里的运行记录，让我们回看今天到底跑了几轮、每轮为什么继续或停止；`memory.md` 是跨天经验，只记录明天还需要注意的问题。也就是说，今天的运行过程一个 Markdown 文件就够了，不需要再做一个跨日期状态库。


#### 2.2.3 再加 gpt-5.4 verifier subagent

&emsp;&emsp;第三步再加 verifier subagent。它对应第一章里的 Sub-agents：我们把“审核日报是否真的合格”这件事交给另一个只读 agent。

&emsp;&emsp;可以发给 Codex：

> 💬 输入给 Codex 的提示词

```text
请为每日 AI 日报任务创建只读 verifier subagent：

路径：
agents/daily-ai-report-verifier.toml

模型：
gpt-5.4

要求：
- 只审核，不生成日报。
- 只读文件，不修改文件, 可以查看原文链接，可以自行执行搜索工具。
- 不提交、不上传、不执行 git push。
- 输入包括当日运行目录、当前轮次、HTML 文件路径、检测脚本结果、source-notes.md、运行日期和 24 小时窗口。

输出必须结构化，可以用 JSON 片段返回给主流程，但不要单独保存成每轮审核文件。主流程读取后，把 decision、reasons、required_fixes 和 quality_score 摘要追加到 runs/YYYY-MM-DD/run-log.md。

输出字段：
{
  "decision": "pass | fix_required | rerun_required",
  "run_date": "YYYY-MM-DD",
  "run_round": N,
  "html_file": "...",
  "reasons": [],
  "required_fixes": [],
  "quality_score": 0-100
}

decision 的含义：
- pass：可以通过。检测脚本不能是 fail，且新闻来源、发布时间、链接、页面结构和视觉规范都可信。
- fix_required：需要局部修复。比如少量措辞不清、图片来源说明不足、某个轻微结构问题，但不需要重新检索新闻。
- rerun_required：需要重跑。比如新闻不足 5 条、多个来源无法确认、使用搜索结果页链接、时间窗口错误或内容不可信。

quality_score 的参考：
- 90-100：质量很好，可以 pass。
- 75-89：基本可用，但通常需要 fix_required，除非只是非常轻微的提醒。
- 50-74：有明显质量问题，通常需要 fix_required 或 rerun_required。
- 0-49：不可靠，通常 rerun_required。

审核重点：
- 新闻是否真实、重要、及时。
- 是否优先使用官方博客、官方文档、GitHub Release、论文页面或公司公告。
- 发布时间、来源、原文 URL 是否已确认。
- “新闻事实”和“为什么重要”是否分开。
- 是否存在无来源图片、无关配图或不确定信息被写成确定事实。
- 页面是否符合 scatterbrain-html-daily 的视觉规范。
- 是否适合阅读、截图和分享。
```


&emsp;&emsp;这里的关键不是说 `gpt-5.4` 比 `gpt-5.5` 更强，而是让审核和生成分离。主流程用 `gpt-5.5` 负责执行和修复，审核者用 `gpt-5.4` 站在验收标准上重新看一遍结果；这仍然是本案例里的成本和独立审核者选择，重点不是模型名本身，而是分离执行者和审核者。

#### 2.2.4 最后把自动化改成 Goal Loop

&emsp;&emsp;最后才改自动化任务 prompt。前面已经准备好了检测脚本、当日运行记录 Skill 和 verifier subagent，这里要做的是把它们接进每天 9 点触发的自动化任务里，让每次运行都按 Goal 推进、检查、修复、记录和停止。


&emsp;&emsp;可以发给 Codex：

> 💬 输入给 Codex 的提示词

```text
请把“每日AI资讯”自动化任务改成 Goal 模式驱动的 loop。

保留：
- 每天 9 点触发。
- 主流程模型使用 gpt-5.5。
- 工作目录：
  当前项目根目录（daily-AI-news-loop）
- 使用 scatterbrain-html-daily Skill 生成中文 HTML 日报。
- 使用 daily-ai-report-run-logger Skill 维护当天 run-log.md。

每次运行时：
1. 读取 automation memory.md；它只提供跨天注意事项，不当作今天的轮次台账。
2. 计算今天日期 YYYY-MM-DD，并确保存在当日目录：
   runs/YYYY-MM-DD/
3. 读取或创建当天运行记录：
   runs/YYYY-MM-DD/run-log.md
   根据已有 Round 小节判断本次是今天第几轮 N。
4. 创建或继续 Goal：
   每日 AI 资讯日报生成与审核
5. 检索过去 24 小时 AI 重要新闻，进入原文确认来源、时间和链接，并把来源核验摘要写入：
   runs/YYYY-MM-DD/source-notes.md
6. 生成 HTML 到：
   runs/YYYY-MM-DD/AI日报-YYYY-MM-DD.html
7. 运行 evals/check_daily_ai_report.py，检查该 HTML，并把最新检测结果写入：
   runs/YYYY-MM-DD/check-result-latest.json
8. 检测 fail 时按错误修复或重跑，最多 2 轮；每轮结束都用 daily-ai-report-run-logger 追加 run-log.md。
9. 检测 pass 或 warn 后，调用 gpt-5.4 的 daily-ai-report-verifier 只读审核。
10. verifier 只返回结构化结论，不另存每轮审核文件；主流程把 decision、reasons、required_fixes 和 quality_score 写入 run-log.md。
11. verifier 不是 pass 时按反馈修复或重跑，最多 2 轮；每轮结束都用 daily-ai-report-run-logger 记录本轮结果、打回原因、修复动作和下一步。
12. 通过或失败时，在 run-log.md 的「今日汇总」里更新总轮数、最终状态、最终 HTML、最终检测结果和最终 verifier 结论。
13. 只有需要第二天继续注意的问题，才写入 automation memory.md。
14. 通过后打开最终 HTML；打不开则输出项目内相对路径。

成功条件：
- HTML 已写入 runs/YYYY-MM-DD/AI日报-YYYY-MM-DD.html。
- 检测脚本 status 为 pass 或 warn。
- verifier decision 为 pass。
- 当日 run-log.md 已记录本轮过程、总轮数和最终状态。
- 浏览器已打开，或已输出 HTML 项目内相对路径。

失败条件：
- 目标目录不可写。
- 无法创建 runs/YYYY-MM-DD/ 当日目录。
- 找不到至少 5 条可靠新闻来源。
- 连续 2 轮修复后检测仍 fail。
- 连续 2 轮修复后 verifier 仍不是 pass。
- 无法确认核心新闻的原文链接、发布时间或来源。
```


&emsp;&emsp;这一步真正做的事，是把 2.1 里“每天 9 点启动一次”的自动化任务，接到一个能收敛的 Goal 上。Automation 负责把任务按时叫醒，Goal 负责推进这一轮；当日目录保存 HTML、source-notes、最新检测结果和 `run-log.md`，运行记录 Skill 负责把每轮结果、verifier 打回原因和总轮数写清楚；`memory.md` 只放需要带到第二天的经验。

&emsp;&emsp;从 Codex 源码看，`/goal` 不是把一句普通消息发给模型。Codex 会先把它写成线程里的 Goal 状态；当 Goal 是 active 且线程空闲时，运行时会注入一段 continuation 指令，让 Codex 继续围绕同一个目标往下做。那段 continuation 指令还要求 Codex 在标记完成前做证据审计；模型自己能调用的 `update_goal` 也只允许标记 `complete` 或 `blocked`，不能随便暂停、恢复或改成预算 / 用量状态。


```text
当日运行目录：保存当天 HTML、source-notes、最新检测结果和 run-log.md。
检测脚本：检查文件、栏目、链接、卡片结构和 scatterbrain 风格线索，给出 pass / warn / fail。
daily-ai-report-run-logger Skill：把主 Agent 每轮做了什么、检测结果、verifier 结论、打回原因和总轮数写入 run-log.md。
verifier subagent：站在独立审核者的位置，看新闻来源、发布时间、重要性和页面质量是否真的过关。
memory.md：只记录需要跨天带走的经验或注意事项。
Goal：把上面这些步骤串成一轮可推进、可修复、可停止的任务。
```


&emsp;&emsp;这也说明了 Goal 的边界：它已经提供了内循环骨架，但不是业务专用验收器，也不会自动替我们设计日报的记录格式。Goal 解决的是“这一轮目标怎么推进、什么时候可以 complete”；检测脚本解决的是“硬性规则有没有过”；当日目录解决的是“当天产物放在哪里”；运行记录 Skill 解决的是“今天跑了几轮、为什么继续或停止”；verifier subagent 解决的是“执行者之外有没有人再验收一次”。如果只用 Goal，不加这些部件，系统仍然容易变成主流程自己判断自己完成。


```text
定时触发
  → 读取 memory.md 里的跨天注意事项
  → 创建或复用 runs/YYYY-MM-DD/ 当日目录
  → 读取或创建 run-log.md，计算今天第 N 轮
  → 围绕 Goal 执行本轮日报生成
  → 检测脚本先检查硬规则，写入最新 check-result-latest.json
  → gpt-5.4 verifier subagent 再审核质量，只返回结构化结论
  → daily-ai-report-run-logger 把本轮结果、打回原因和修复动作写入 run-log.md
  → 根据脚本和审核反馈修复或重跑
  → 通过后 complete，失败时在 run-log.md 留下原因
  → 需要第二天注意的问题另写入 memory.md
```


&emsp;&emsp;然后我们执行一下

> 💬 输入给 Codex 的提示词

```text
跑一下这个自动化吧
```

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch02-daily-ai-original-06-2ab180b2.png" width=80%></div>

&emsp;&emsp;可以看到它确实启动了 Goal 模式，不用我们手动使用 `/goal` 命令来启动

&emsp;&emsp;这就和第一章的定义对齐了：Loop Engineering 是 harness 之上的外层控制系统。在 Codex 里，Automation、Skill、Sub-agent、运行记录、必要时的 Worktree 和 plugins/connectors，提供了 loop 的外层部件；检测脚本、Goal、当日 `run-log.md` 和停止条件，则把这个日报 loop 进一步约束到“可验收、可复盘、可停止”。它们共同构成了本案例里的 loop engineering。


## <center>第三章：手动构建 Loop Engineering</center>

&emsp;&emsp;第二章里，我们已经看到每日 AI 资讯任务如何借助 Automation、Skill、当日运行记录、verifier 以及 Codex 自带的 Goal 模式，拼成相对完整的 loop engineering。相比定时触发的“外循环”，更难的是建立一个能朝着目标推进、接受检查、根据反馈继续修正的“内循环”。所以第三章的目标，是在没有内置 Goal 模式的情况下，手动搭出一个可检查、可复核、可停止的框架迁移 loop engineering。


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/embedded-cell-151-c1682aa7.png" width=80%></div>


### 3.1 手动构建 Loop Engineering

&emsp;&emsp;本案例模拟一个真实工程场景：我们接手了一个陌生的旧项目，需要在尽量不破坏原有功能和外部接口约定的前提下，完成一次底层框架<font color=red>迁移</font>。这个旧项目是一个基于 CrewAI 的<b>销售数据报表生成系统</b>，迁移目标是将它的底层 Agent 编排框架从 <b>CrewAI</b> 切换为 <b>LangGraph</b>。这里真正要展示的，不是 CrewAI 和 LangGraph 两个框架之间的简单替换，而是当我们面对一个陌生项目时，如何用 <b>loop engineering</b> 的方式，让 Codex 先固定基线、隔离工作区、理解旧系统，再通过健康检查、修改和 verifier 复核，完成一次可控的框架迁移。


&emsp;&emsp;工程边界很明确：业务功能、前端展示、数据库结构、对外 API、SSE 事件流和 Workflow 关键行为都保持兼容，主要变化只发生在后端 Agent 编排框架。


&emsp;&emsp;这里需要先说明一个关键点：本案例不依赖 <font color=red>Codex</font> 或其他 Agent 产品内置的 <b>Goal 模式</b>。很多 Agent 产品并没有 Goal 模式。即使有，Goal 模式的执行过程也不一定足够透明，可能很难明确看到每一轮的目标、检查结果、剩余问题和停止依据。所以本案例要展示的是另一种更通用的做法：<b>不用产品内置 Goal 模式，而是用工程化资产手动构建一个类似 Goal 模式的 loop engineering 流程</b>。也就是说，我们不把“持续循环”完全交给 Agent 自己记忆，而是把循环拆成外部可见、可检查、可复核的几个部分：


&emsp;&emsp;通用的构建顺序可以概括为：


<div align="center">
<table width="80%">
<thead><tr><th>步骤</th><th>通用目的</th><th>本案例里的落地</th></tr></thead>
<tbody>
<tr><td>固定基线</td><td>让后续修改可回滚、可对照</td><td>初始化 Git baseline，创建迁移 worktree</td></tr>
<tr><td>只读理解旧系统</td><td>先确认真实运行流程，而不是猜框架用法</td><td>分析 CrewAI 项目的 <code>/api/run</code>、SSE、数据库和 workflow</td></tr>
<tr><td>形成迁移地图</td><td>把旧行为、目标设计和边界写成外部依据</td><td><code>docs/langgraph_migration_map.md</code></td></tr>
<tr><td>建立静态检查</td><td>自动发现结构、接口约定和边界破坏</td><td><code>evals/check_migration_health.py</code></td></tr>
<tr><td>建立运行时冒烟检查</td><td>证明服务能启动、关键 API 和流式路径能跑通</td><td><code>evals/check_runtime_smoke.py</code></td></tr>
<tr><td>建立真实任务测试集</td><td>证明不同输入仍产生符合预期的关键行为</td><td><code>evals/query_smoke_cases.json</code></td></tr>
<tr><td>建立状态记录</td><td>让每轮 round 的目标、结果和剩余问题可追踪</td><td><code>docs/langgraph_migration_state.md</code></td></tr>
<tr><td>配置只读复核</td><td>用独立 verifier 判断 complete / fix_required</td><td><code>agents/langgraph-migration-verifier.toml</code></td></tr>
<tr><td>固化主循环协议</td><td>让主 Agent 每轮按固定流程执行</td><td><code>AGENTS.md</code></td></tr>
<tr><td>执行迁移循环</td><td>修改、检查、修复、复核，直到满足完成条件</td><td><code>worktrees/langgraph</code> 迁移工作区中持续推进</td></tr>
</tbody>
</table>
</div>


```text
Git baseline / worktree：固定基线并隔离实验环境
migration map：记录旧系统行为、目标设计和迁移边界
health check：提供每轮自动化反馈
migration task cases：用少量真实任务检查语义是否退化
migration state：记录当前轮次、检查结果和剩余问题
verifier：在每轮结束后做只读复核
AGENTS.md：约束主 Agent 的执行协议
```

&emsp;&emsp;这里说的“<font color=red>手动构建</font>”，并不是指只靠聊天完成循环。单纯对话式 Agent 不能写文件、不能运行脚本、不能读取检查结果，无法真正形成工程闭环。本案例默认使用的是<b>具备工具调用能力的 coding Agent</b>：它可以读写项目文件、执行 Shell 命令、使用 Git/worktree、运行健康检查脚本，并在需要时调用 verifier subagent。在这个前提下，即使没有产品内置 Goal 模式，coding Agent 也可以通过外部工程资产形成一个接近 Goal 模式的闭环：


```text
读取目标和边界
  ↓
执行一轮迁移尝试
  ↓
运行健康检查
  ↓
如果修改了运行路径/API/SSE/依赖，则启动服务并跑 runtime smoke test
  ↓
根据 FAIL / WARN 修复
  ↓
更新状态文件
  ↓
交给 verifier 判断 complete / fix_required
  ↓
如果未完成，进入下一轮
```

&emsp;&emsp;对于agent接手陌生项目来说，最大的问题是它一开始并不知道旧系统真正是怎么运行的。项目表面上使用了 CrewAI，但 CrewAI 是否真的负责完整 workflow？数据库查询是 Agent 调工具完成，还是普通 Python 代码执行？SSE 事件由哪个模块发出？前端依赖哪些字段？这些都不能靠猜。所以，我们不能直接输入一句：


```text
帮我把这个 CrewAI 项目迁移成 LangGraph。
```

&emsp;&emsp;这样容易让 Agent 一次性大改代码，表面完成迁移，却破坏 API、SSE、数据库结构或前端依赖。本案例的重点，就是把迁移放进“目标明确、环境隔离、指标可测、反馈可用”的工程闭环中。


### 3.2 隔离实验环境：准备 Git baseline 与迁移 worktree

&emsp;&emsp;明确了本案例采用的是“手动构建 loop engineering”之后，第一步不是让 <font color=red>Codex</font> 修改代码，而是先固定迁移前状态，并创建独立的迁移工作区。在正式让 Codex 执行迁移前，我们不会直接让它修改原项目。第一步是把当前 CrewAI 项目初始化成 Git 仓库，并提交一个 baseline commit。这个 commit 代表“迁移前的确定状态”。然后基于这个 baseline 创建一个独立的 Git worktree，比如：


```text
当前项目根目录：原始 CrewAI baseline

worktrees/langgraph：LangGraph 迁移工作区
```


&emsp;&emsp;这里要注意，<font color=red>worktree</font> 不是简单复制一份文件夹。它仍然属于同一个 Git 仓库，只是把不同分支 checkout 到不同目录中。在本案例中，原始目录保留在 `main` 分支，作为稳定的 CrewAI baseline；新的 worktree 目录处在 `migration/langgraph-loop` 分支，后续所有 LangGraph 迁移都在这个目录里完成。这样做可以同时满足可对照、可回滚、可验证：原始 baseline 保持稳定，迁移 worktree 用来运行后续修改、测试和健康检查。接下来可以新开一个 Codex 会话窗口，将下面的提示词输入给 Codex，让它只完成 Git baseline 和 worktree 准备，不开始迁移代码：


> 💬 输入给 Codex 的提示词

```text
请先不要迁移代码，只做 Git baseline 和 worktree 准备。

项目路径：
当前项目根目录

任务：
1. 在该项目中初始化 Git 仓库。
2. 检查并完善 .gitignore，确保不要纳入：
   - .env
   - .venv/
   - __pycache__/
   - backend/data/*.db
   - backend/logs/
   - .DS_Store
   - evals/results/
   - worktrees/
3. 将当前项目源码作为 CrewAI baseline 纳入 Git。
4. 创建 baseline commit：
   commit message: baseline: crewai workflow sales report
5. 基于 baseline 创建迁移 worktree：
   - 分支名：migration/langgraph-loop
   - worktree 路径：
     worktrees/langgraph
6. 不要迁移代码。
7. 不要创建测试脚本。
8. 不要运行 benchmark。

完成后请汇报：
- Git repo 路径（项目内相对路径即可）
- baseline commit hash
- worktree 路径（项目内相对路径）
- worktree 分支名
- 原始 repo 的 git status
- worktree 的 git status
- .gitignore 当前内容
```


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch03-migration-original-01-ee922fd0.png" width=80%></div>

&emsp;&emsp;Codex 已经基于原始项目创建了新的迁移 worktree，当前形成两个工作目录：


```text
当前项目根目录：原始 CrewAI baseline，对应 main 分支

worktrees/langgraph：LangGraph 迁移工作区，对应 migration/langgraph-loop 分支
```


&emsp;&emsp;第一步只固定迁移前状态并准备独立迁移环境，没有开始迁移，也没有修改业务代码。下一步，我们会进入迁移 worktree，让 Codex 先识别旧系统真实的 Workflow，再提出候选 LangGraph 迁移设计。


### 3.3 理解旧系统：识别真实行为与迁移边界

&emsp;&emsp;完成 Git baseline 和 worktree 准备后，我们已经有了一个独立的迁移工作区：worktrees/langgraph


&emsp;&emsp;接下来还不能直接让 Codex 迁移代码。CrewAI 和 LangGraph 的编排概念并不一一对应，所以这一节先让 Agent 在隔离 worktree 中识别旧系统真实流程，再提出后续 loop 可用的依据：


```text
旧 Workflow：当前系统真实运行流程
候选设计：LangGraph node / State / Edge 的初步方案
迁移边界：哪些地方可以改，哪些地方不能随便改
健康检查：后续每一轮 loop 后应该检查什么
```

&emsp;&emsp;这一节不一次性索要完整方案，而是分轮提问，每轮只解决一个问题，并把结果作为下一轮依据。这一节可以拆成四轮：


```text
第一轮：识别旧系统真实 Workflow
第二轮：提出候选 LangGraph 迁移设计
第三轮：识别迁移边界
第四轮：提出后续 loop 健康检查项
```

#### 3.3.1 识别真实运行流程：旧系统 workflow

&emsp;&emsp;第一轮先不问 <font color=red>LangGraph</font>，也不设计迁移方案，只解决一个问题：当前 CrewAI 项目真实是怎么运行的？很多项目虽然用了 CrewAI，但真实流程可能仍然是普通 Python 顺序控制，CrewAI 只负责其中几次 LLM 调用。所以第一轮要先做只读分析。可以将下面的提示词输入给 Codex：


> 💬 输入给 Codex 的提示词

```text
请在迁移 worktree 中只读分析当前项目，不要修改任何代码。

项目路径：
当前迁移 worktree 根目录

当前任务：
我们要用 loop engineering 将项目从 CrewAI 迁移到 LangGraph。
现在先不要设计 LangGraph，也不要提出迁移方案。
请你只分析旧系统真实的 workflow 是如何运行的。

重点回答：
1. 旧系统的主执行入口在哪里？
2. 用户请求进入后，完整执行顺序是什么？
3. 哪些步骤由 CrewAI 完成？
4. 哪些步骤由普通 Python 代码完成？
5. 哪些步骤属于 FastAPI/server 层逻辑？
6. 每个阶段的输入和输出是什么？
7. CrewAI 在当前项目里到底是完整 workflow 编排者，还是只负责部分 LLM 单步任务？

要求：
- 不要修改代码。
- 不要创建文件。
- 不要安装依赖。
- 不要开始迁移。
- 只输出旧 workflow 分析。
- 最后给出一张“旧 workflow 阶段表”。
```


&emsp;&emsp;<font color=red>Codex</font> 第一轮返回的关键判断是：<b>旧系统真正的 workflow 编排入口不是 CrewAI 的自动 sequential 编排，而是 `backend/agent.py` 中的 `run_events(topic)`</b>。CrewAI 在当前项目里只承担若干 LLM 单步任务，例如生成 SQL、修正 SQL、统计解读、图表配置和报告撰写。

&emsp;&emsp;因此，后续要迁移的不是 CrewAI 调用本身，而是旧系统中已经存在的完整 workflow 行为。这里放 Codex 的第一轮返回结果截图：


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch03-migration-original-02-a93f5042.png" width=80%></div>

&emsp;&emsp;<font color=red>旧系统大致分为三层</font>：

&emsp;&emsp;第一层是 <b>FastAPI / server 层</b>，负责接收 `/api/run` 请求、建立 SSE 连接、拼接历史上下文、发送 `final` 事件并保存历史。第二层是 <b>Python workflow 层</b>，负责数据源准备、SQL 清洗、SQL 执行、失败重试、图表解析兜底和事件发射。第三层是 <b>CrewAI 单步任务层</b>，负责 SQL 生成、SQL 修正、统计解读、图表配置文本和报告撰写。


&emsp;&emsp;CrewAI 并不是当前项目的完整 workflow 编排者，它更像是被 Python workflow 调用的 LLM 执行单元。后续迁移要保留的不只是“能生成报告”，还包括：


```text
/api/run 请求入口
SSE 事件流
历史上下文拼接
数据源准备
SQL 生成与执行
SQL 失败后最多一次重试
统计分析
图表生成与兜底解析
Markdown 报告生成
final 事件与历史保存
```

&emsp;&emsp;这些内容会成为后续候选设计和健康检查的依据。


#### 3.3.2 设计候选目标结构：LangGraph State / Node / Edge

&emsp;&emsp;有了旧 workflow 之后，第二轮才开始让 <font color=red>Codex</font> 思考 LangGraph。这一轮也不修改代码，只让 Codex 回答：如果用 LangGraph 等价表达旧 workflow，应该如何设计 State、Node 和 Edge？这不是 CrewAI API 到 LangGraph API 的机械翻译，而是让 Codex 根据旧系统真实行为阶段提出候选设计。


&emsp;&emsp;可以继续输入下面的提示词：


> 💬 输入给 Codex 的提示词

```text
基于你刚才识别出的旧 workflow，请继续只读分析，不要修改任何代码。

当前任务：
请不要做代码迁移，只提出一个候选的 LangGraph 迁移设计。

注意：
这不是 CrewAI API 到 LangGraph API 的机械翻译。
请根据旧 workflow 中真实存在的行为阶段，设计 LangGraph 的 State、Node 和 Edge。

请重点回答：
1. 旧 workflow 中哪些行为阶段适合拆成 LangGraph node？
2. 哪些逻辑应该保留在 FastAPI/server 层，不应该放进 graph？
3. State 中需要保存哪些字段？
4. 哪些分支需要用 conditional edge 表达？
5. 为什么不能把旧 run_events() 整体包成一个 LangGraph node？

要求：
- 不要修改代码。
- 不要创建文件。
- 不要开始迁移。
- 输出一张“旧 workflow 阶段到 LangGraph node 的候选设计表”。
- 表格至少包含：
  - 旧行为阶段
  - 候选 LangGraph node
  - 读取 State
  - 写入 State
  - 设计理由
```

&emsp;&emsp;Codex 第二轮返回后，先把旧系统执行链路概括为：


```text
source → query LLM → extract SQL → run SQL → optional repair → stats LLM → chart LLM → parse chart → report LLM
```

&emsp;&emsp;候选 LangGraph node 围绕这些行为阶段设计：


```text
load_source
generate_sql
execute_sql
repair_sql
analyze_stats
generate_chart
write_report
```

&emsp;&emsp;同时，并不是所有逻辑都应该进入 LangGraph。`/api/run` 路由、SSE 连接、历史会话、`start/final/error` 包装事件、CORS、静态文件和数据库浏览接口都更适合保留在 FastAPI / server 层。


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch03-migration-original-03-d92d873c.png" width=80%></div>

&emsp;&emsp;候选设计把旧 workflow 的真实行为阶段拆成更明确的 node：


- 数据源准备可以设计为 `load_source`；
- SQL 生成可以设计为 `generate_sql`；
- SQL 执行可以设计为 `execute_sql`；
- SQL 报错后的修正可以设计为 `repair_sql`；
- 统计分析可以设计为 `analyze_stats`；
- 图表生成可以设计为 `generate_chart`；
- 报告撰写可以设计为 `write_report`；
- `final` SSE 和历史保存继续保留在 server 层。


&emsp;&emsp;重点不是 node 名字本身，而是把旧 workflow 中的状态流转和条件分支显式化。例如 SQL 执行失败后的修复逻辑应该表达为：


```text
generate_sql → execute_sql → 判断是否成功
                          ↓
                    失败则 repair_sql
                          ↓
                    再次 execute_sql
```

&emsp;&emsp;这个分支就是 LangGraph 里适合使用 conditional edge 的地方。不能把旧的 `run_events()` 整体包成一个 LangGraph node，否则 SQL 生成、执行、重试、统计、图表和报告阶段仍然藏在一个大函数里，LangGraph 看不到这些阶段，也就无法做到：


```text
单独观察每个阶段
单独验证每个状态
单独表达 SQL retry 分支
单独做 checkpoint、重放和局部重试
```

&emsp;&emsp;否则就会变成“套壳迁移”：代码能跑，但迁移目标没有真正完成。第二轮得到的结果本质上是一份候选迁移设计：


```text
旧行为阶段 → 候选 LangGraph node → State 读写 → 设计理由
```

&emsp;&emsp;它不是最终答案，但已经为后面的 loop 提供了目标结构：每一轮迁移都围绕这些 node、State 和 conditional edge 推进。


#### 3.3.3 识别迁移边界：主要修改范围与兼容接口约定

&emsp;&emsp;有了候选<font color=red>迁移</font>设计之后，第三轮要识别迁移边界：后续 loop 中哪些地方可以改，哪些地方不能随便改？框架迁移很容易扩大范围。项目能跑不代表迁移成功，如果 API、SSE、数据库或前端接口约定被破坏，就已经偏离目标。继续输入下面的提示词：


> 💬 输入给 Codex 的提示词

```text
继续只读分析，不要修改任何代码。

当前任务：
请根据旧 workflow 和候选 LangGraph 设计，识别后续迁移 loop 的边界。

请重点回答：
1. 哪些文件是主要迁移对象？
2. 哪些文件原则上不应该修改？
3. 哪些 API 契约必须保持兼容？
4. 哪些 SSE 事件名、事件顺序、payload 字段必须保持兼容？
5. 哪些数据库和前端契约不能破坏？
6. 如果后续必须修改边界外文件，应该如何说明和确认？

要求：
- 不要修改代码。
- 不要创建文件。
- 不要开始迁移。
- 最后给出一张“迁移边界表”。
```

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch03-migration-original-04-1725342e.png" width=80%></div>

&emsp;&emsp;后续迁移的主要修改范围集中在：


```text
backend/agent.py
requirements.txt
```

&emsp;&emsp;`backend/agent.py` 是核心迁移对象；`requirements.txt` 用于替换依赖。`backend/server.py` 只在必要时做薄适配，继续保留 `/api/run`、SSE、历史记录、final 保存等 server 层职责。而下面这些文件原则上不应该随便修改：


```text
frontend/index.html
backend/tools.py
backend/runtime.py
backend/store.py
backend/seed.py
backend/config.py
backend/app.py
```

&emsp;&emsp;后续每一轮迁移都要保持这些<b>关键行为不变</b>：


```text
API 路径不变
SSE 事件结构不变
数据库 schema 不变
前端消费方式不变
历史记录格式不变
旧 workflow 的关键行为尽量等价保留
```

&emsp;&emsp;如果后续必须修改边界外文件，也不能直接改，而是要先说明原因、影响范围和兼容策略，再决定是否修改。这样后续 loop 才是受控迁移。


#### 3.3.4 设计反馈检查项：后续 round 要验证什么

&emsp;&emsp;第四轮要和<font color=red>下一小节</font>衔接起来。现在我们已经知道旧系统怎么运行，也有了候选 LangGraph 设计和迁移边界。接下来要问 Codex：后续每一轮迁移后，应该检查什么？这一步不是马上写测试脚本，而是先让 Codex 给出健康检查清单。第三小节再把其中最关键、最适合自动化的部分落成脚本。


&emsp;&emsp;继续输入下面的提示词：


> 💬 输入给 Codex 的提示词

```text
继续只读分析，不要修改任何代码。

当前任务：
请根据旧 workflow、候选 LangGraph 设计和迁移边界，提出后续 loop engineering 中每一轮迁移后应该运行的健康检查。

请重点回答：
1. 哪些检查可以自动化？
2. 哪些检查需要人工或 verifier 复核？
3. 如何检查不是“把旧 run_events() 包成一个 LangGraph node”的套壳迁移？
4. 如何检查 CrewAI 是否仍在运行路径中？
5. 如何检查 API、SSE、数据库、前端契约没有被破坏？
6. 如何检查 SQL retry、chart fallback、历史上下文这些旧行为仍然保留？

要求：
- 不要修改代码。
- 不要创建文件。
- 不要开始迁移。
- 最后给出一张“后续 loop 健康检查建议表”。
```

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch03-migration-original-05-c1a79ecf.png" width=80%></div>

&emsp;&emsp;从 Codex 的返回结果可以看到，后续健康检查主要分成几类：


```text
Graph 结构检查
CrewAI 依赖检查
API / SSE 契约检查
数据库契约检查
旧行为回归检查
人工或 verifier 复核
```

&emsp;&emsp;这里最重要的不是检查项有多少，而是它们会成为后续 loop 的反馈来源。也就是说，后续每一轮迁移不能只问 Codex：


```text
你改好了吗？
```

&emsp;&emsp;而是要让它根据检查结果继续推进：


```text
这轮改动有没有超出边界？
Graph 是不是真的拆成了多个 node？
有没有把旧 run_events() 整体包成一个 node？
CrewAI 是否还在运行路径中？
API、SSE、数据库、前端契约有没有被破坏？
SQL retry、chart fallback、历史上下文这些旧行为是否还在？
```

&emsp;&emsp;尤其要注意“套壳<font color=red>迁移</font>”这个问题。如果只是把旧的 `run_events()` 包进一个 LangGraph node，项目可能也能运行，但 LangGraph 并没有真正接管 workflow。旧系统里的 SQL 生成、SQL 执行、失败重试、统计分析、图表生成、报告撰写，仍然被藏在一个大函数里，后续也无法按 node 粒度检查、回放和修复。所以健康检查的意义，就是把“迁移是否完成”从主观判断变成可验证条件。这一轮完成后，我们就有了后续 loop 的反馈机制：


```text
修改
  ↓
运行健康检查
  ↓
根据失败项诊断
  ↓
修复后再次检查
  ↓
进入下一轮
```

&emsp;&emsp;下一小节就可以基于这些检查项，先建立一套迁移健康检查，让后续每一轮修改都有明确的验证依据。


#### 3.3.5 阶段产物：形成迁移地图

&emsp;&emsp;这一节的产出不是迁移代码，而是后续 loop 的控制依据：


```text
旧 Workflow：说明旧系统真实怎么跑
候选设计：说明 LangGraph 可以如何表达旧行为
迁移边界：说明哪些文件和契约不能随便破坏
健康检查：说明每轮迁移后要检查什么
```

&emsp;&emsp;为了避免这些结论只停留在聊天记录里，需要把前面四轮只读分析整理成一份迁移地图文档。后续每一轮 loop 都先读取这份文档，再决定本轮改什么、哪些边界不能碰、应该运行哪些检查。


&emsp;&emsp;可以继续输入下面的提示词：


> 💬 输入给 Codex 的提示词

```text
请不要迁移代码，也不要修改业务文件。

请根据前面四轮只读分析结果，整理一份迁移地图文档：

docs/langgraph_migration_map.md

文档需要包含：
1. 旧系统真实 Workflow；
2. CrewAI 在旧系统中的真实职责；
3. 候选 LangGraph node / State / Edge 设计；
4. 不应进入 graph 的 FastAPI/server 层逻辑；
5. 迁移边界：主要修改文件、薄适配文件、原则上不改文件；
6. 必须保持兼容的 API / SSE / DB / frontend 契约；
7. 后续 loop 健康检查项；
8. 当前未决假设和需要 verifier 复核的问题。

要求：
- 不要迁移业务代码；
- 不要修改前端；
- 不要替换 CrewAI；
- 只创建或更新 docs/langgraph_migration_map.md；
- 完成后汇报文档路径、主要章节和 git status。
```

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch03-migration-original-06-7da7e4e4.png" width=80%></div>

&emsp;&emsp;Codex 执行完成后，只创建了迁移地图文档，没有修改业务代码、前端，也没有替换 CrewAI。文档主要包含：


```text
旧系统真实 Workflow
CrewAI 在旧系统中的真实职责
候选 LangGraph Node / State / Edge 设计
不应进入 Graph 的 FastAPI / server 层逻辑
迁移边界
必须保持兼容的 API / SSE / DB / Frontend 契约
后续 Loop 健康检查项
当前未决假设和需要 Verifier 复核的问题
```

&emsp;&emsp;当前 git status 显示：


```text
## migration/langgraph-loop
?? docs/langgraph_migration_map.md
```

&emsp;&emsp;接下来把迁移地图单独提交，便于后续区分“分析产物”和“迁移实现改动”。可以继续输入下面的提示词：


> 💬 输入给 Codex 的提示词

```text
请只提交刚刚生成的迁移地图文档，不要修改任何业务代码。

请确认当前分支是：

migration/langgraph-loop

请执行：
1. git status --short
2. 确认只有 docs/langgraph_migration_map.md 是新增文件
3. git add docs/langgraph_migration_map.md
4. git commit -m "docs: add langgraph migration map"
5. 提交后汇报：
   - commit hash
   - git status --short --branch

如果发现除 docs/langgraph_migration_map.md 之外还有其他未提交改动，请停止并说明。
```

&emsp;&emsp;提交完成后，第二步结束：迁移地图已经作为独立 commit 固化。下一步基于这份地图建立健康检查、状态文件和 verifier 规则。


### 3.4 建立反馈机制：检查、状态与复核资产

&emsp;&emsp;前面已经把旧系统分析、候选设计、迁移边界和检查项固化到迁移地图：


```text
docs/langgraph_migration_map.md
```

&emsp;&emsp;第三步要把手动 loop 落成具体工程资产，让每一轮 round 都有状态记录、自动化检查和独立复核。这里的 round 指主 Agent 围绕总目标完成一次完整迁移尝试，结束时再把最终检查结果、修改摘要和剩余问题写入状态文件，并交给 verifier 判断。这一节要准备五类工程资产：


```text
evals/check_migration_health.py
自动化健康检查脚本，负责在一轮内部和一轮结束前提供检查反馈。

evals/check_runtime_smoke.py
运行时冒烟测试脚本，负责在涉及后端运行路径的 round 中启动服务，并验证关键 API / SSE 是否真实可用。

docs/langgraph_migration_state.md
迁移状态文件，负责记录每轮 round 的开始计划、最终结果、检查状态和下一步计划。

agents/langgraph-migration-verifier.toml
verifier subagent 配置，负责在一轮结束后判断 complete / fix_required。

AGENTS.md
主 Agent 的项目级 loop 协议，规定每轮怎么执行、怎么更新状态、什么时候停止。
```


&emsp;&emsp;`migration_map.md` 是上一节的产物；本节新增的健康检查、runtime smoke test、状态文件、verifier 和 `AGENTS.md`，共同把“继续迁移”变成有反馈、有记录、有复核的 loop：


```text
读取 migration_map.md 和 migration_state.md
  ↓
写入本轮 round 的开始计划
  ↓
主 Agent 围绕总目标完整尝试一次迁移
  ↓
过程中可以多次运行健康检查并自行修复
  ↓
本轮结束前运行最终健康检查
  ↓
如果本轮修改了后端运行路径、API、SSE、依赖或 graph 代码，则启动服务并运行最小 smoke test
  ↓
把修改结果、健康检查结果、剩余问题写入 migration_state.md
  ↓
调用 verifier subagent 判断 complete / fix_required
  ↓
如果 complete，停止；如果 fix_required，进入下一轮 round
```

&emsp;&emsp;<font color=red><code>max_rounds</code></font> 是安全上限。比如先设置为 6，如果 6 轮后仍未完成，就停止并汇报阻塞点。

&emsp;&emsp;开始建立反馈机制前，先确认当前确实位于迁移 worktree 中，避免把辅助文件写回原始 CrewAI baseline。可以先输入下面的提示词：


> 💬 输入给 Codex 的提示词

```text
请先只做环境确认，不要创建文件，不要修改代码。

请确认当前操作目录是否为当前迁移 worktree 根目录

并确认当前 Git 分支是否为：

migration/langgraph-loop

请执行并汇报：

1. 当前工作目录：
   pwd

2. 当前 Git 分支和状态：
   git status --short --branch

3. 当前 worktree 列表：
   git worktree list

要求：
- 不要修改任何文件；
- 不要创建任何文件；
- 不要开始迁移；
- 如果当前目录不是迁移 worktree 根目录，或者分支不是 migration/langgraph-loop，请停止并说明问题，不要继续后续步骤。
```


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch03-migration-original-07-9edc695d.png" width=80%></div>

&emsp;&emsp;Codex 确认当前已经位于迁移 worktree：当前迁移 worktree 根目录


&emsp;&emsp;当前分支也是：migration/langgraph-loop


&emsp;&emsp;后续健康检查脚本、运行时冒烟脚本、真实任务测试集、状态文件、verifier 配置和 `AGENTS.md` 都会写在迁移工作区中。


#### 3.4.1 建立静态健康检查：目标、约束、回归与证据

&emsp;&emsp;首先需要创建<b>自动化<font color=red>健康检查</font>脚本</b>。健康检查脚本是后续循环的<b>反馈传感器</b>。它不直接判断“整个迁移是否完成”，而是在每一轮 round 中提供外部反馈。前面我们已经生成了迁移地图：


```text
docs/langgraph_migration_map.md
```

&emsp;&emsp;迁移地图已经记录了旧 Workflow、候选 <font color=red>LangGraph</font> 设计、迁移边界和健康检查项，所以这一轮是把这些判断依据落成脚本。健康检查脚本的价值，是把“Agent 觉得自己改好了”变成“有外部检查结果作为反馈”。


&emsp;&emsp;<b>健康检查的通用设计方法</b>

&emsp;&emsp;健康检查围绕四个问题设计：


<div align="center">
<table width="80%">
<thead><tr><th><b>检查类型</b></th><th><b>核心问题</b></th><th><b>本案例中的例子</b></th></tr></thead>
<tbody>
<tr><td><b>目标达成检查</b></td><td>当前结果离目标更近了吗？</td><td>是否逐步出现 LangGraph node / State / Edge，而不是旧 <code>run_events()</code> 套壳</td></tr>
<tr><td><b>约束遵守检查</b></td><td>本轮有没有违反事先规定的限制？</td><td>API 路径、请求参数、返回字段、SSE 事件名、数据库表结构、前端依赖字段是否保持兼容，是否改了不该改的文件</td></tr>
<tr><td><b>质量回归检查</b></td><td>原来重要的质量和能力有没有下降？</td><td>SQL retry、chart fallback、历史上下文、只读 SQL 安全边界是否仍然保留</td></tr>
<tr><td><b>证据充分检查</b></td><td>判断“完成 / 继续 / 修复”有没有足够证据？</td><td>是否运行健康检查、展示 PASS / WARN / FAIL、diff 摘要和下一步计划</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;在本案例中，健康检查脚本就应该围绕这些内容生成：


```text
Graph 是否真实拆成 node / State / Edge
是否存在套壳迁移
CrewAI 是否仍在运行路径中
API 路径、请求参数、返回字段是否保持兼容
SSE 事件名、事件顺序、payload 字段是否保持兼容
数据库表结构和只读 SQL 安全边界是否保持不变
前端依赖字段是否还能正常消费
SQL retry、chart fallback、历史上下文是否保留
```

&emsp;&emsp;这里的“兼容”指迁移前后外部使用方式不能变化。当前前端通过 `EventSource` 监听 `/api/run` 返回的 SSE 事件，迁移后需要保持这些事件名和关键 payload 字段兼容：


```text
start
source
stage_start
stage_retry
sql
query_result
stage_done
stats
chart
final
error
```

&emsp;&emsp;`final` 事件里原来包含：


```text
report
chart
sql
columns
rows
```

&emsp;&emsp;迁移后必须保持 `/api/run` 的 SSE 接口约定兼容，尤其是 `final` 事件中的 `report`、`chart`、`sql`、`columns`、`rows`。健康检查脚本输出三种状态：


<div align="center">
<table width="80%">
<thead><tr><th><b>状态</b></th><th><b>含义</b></th><th><b>后续动作</b></th></tr></thead>
<tbody>
<tr><td><b>PASS</b></td><td>当前检查项通过，没有发现问题</td><td>可以作为本轮继续推进或提交 verifier 的证据</td></tr>
<tr><td><b>WARN</b></td><td>当前阶段允许存在，但后续必须处理或确认</td><td>不阻断当前 round，但要写入状态文件，后续跟踪</td></tr>
<tr><td><b>FAIL</b></td><td>当前检查项失败，说明已经破坏目标、边界或关键行为</td><td>必须在本轮内先修复，不能交给 verifier 作为完成证据</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;当前还没有正式开始 LangGraph 迁移，所以部分目标还不能直接判定失败。例如：


```text
是否已经存在多个 LangGraph node
是否已经存在 conditional edge
CrewAI 是否已经完全移除
```

&emsp;&emsp;这些目标当前可以先标记为 `WARN`，等进入正式迁移后再逐步收紧为 `FAIL`。可以输入下面的 prompt：


> 💬 输入给 Codex 的提示词

```text
请基于 docs/langgraph_migration_map.md，创建后续迁移 loop 使用的自动化健康检查脚本。

要求：
1. 不要开始迁移业务代码。
2. 不要替换 CrewAI。
3. 不要修改前端。

请创建：

evals/check_migration_health.py

脚本用途：
用于后续每一轮 round 中的自动化健康检查。
主 Agent 可以在一轮 round 内多次运行它来发现问题、修复问题。
在一轮 round 结束、准备交给 verifier 判断前，必须运行一次最终健康检查，并把结果写入状态文件。

检查项请从 docs/langgraph_migration_map.md 中提取，并按照四类组织：

1. 目标达成检查
   - 是否真实推进 LangGraph node / State / Edge 迁移；
   - 是否存在把旧 run_events() 套成一个大 node 的表面迁移。

2. 约束遵守检查
   - API 路径、请求参数、返回字段是否保持兼容；
   - SSE 事件名、事件顺序、payload 字段是否保持兼容；
   - 数据库表结构和只读 SQL 安全边界是否保持不变；
   - 前端依赖字段是否还能正常消费；
   - 是否修改了原则上不该动的文件。

3. 质量回归检查
   - SQL retry、chart fallback、历史上下文等旧行为是否保留；
   - 只读 SQL 安全边界是否仍然存在。

4. 证据充分检查
   - 是否输出清晰的 PASS / WARN / FAIL；
   - 是否方便后续 Codex 和 verifier 根据失败项继续修复或判断。

PASS / WARN / FAIL 判定规则：

- PASS：
  当前检查项通过，没有发现问题。

- WARN：
  当前阶段允许存在，但后续必须处理或确认。
  WARN 不应该让脚本返回非 0。
  例如：当前还没正式迁移，所以 CrewAI 仍存在、LangGraph node 尚未出现、conditional edge 尚未出现，可以先 WARN。

- FAIL：
  当前检查项失败，说明已经破坏目标、边界或关键行为。
  FAIL 必须让脚本返回非 0。
  例如：/api/run 消失、final payload 关键字段缺失、SSE 事件名缺失、tools.run_sql 不再限制 SELECT、前端文件被无理由修改。

脚本输出要求：
- 每个检查项输出 PASS / WARN / FAIL；
- 每个 WARN / FAIL 需要给出原因；
- 最后输出 PASS / WARN / FAIL 汇总；
- 如果存在 FAIL，用非 0 退出码；
- 如果只有 WARN，不要阻断；
- 输出要清楚，方便后续 Codex 根据失败项继续修复。

完成后请运行一次：

python3 evals/check_migration_health.py

并汇报：
1. 新增了哪些文件；
2. 当前 PASS / WARN / FAIL 汇总结果；
3. 请输出一张“健康检查项清单表”，包含：
   - 检查分类；
   - 检查项名称；
   - 检查目的；
   - 当前结果：PASS / WARN / FAIL；
   - 当前原因；
   - 后续 loop 中的作用。
4. 当前有哪些 WARN；
5. 这些 WARN 为什么现在不阻断；
6. 后续哪些 WARN 应该在迁移完成前变成 PASS；
7. 是否有 FAIL；
8. 是否修改了业务代码、前端或 CrewAI 依赖；
9. git status。
```

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch03-migration-original-08-6dc96dc8.png" width=80%></div>

&emsp;&emsp;Codex 执行完成后，会新增自动化健康检查脚本：


```text
evals/check_migration_health.py
```

&emsp;&emsp;随后它会运行一次健康检查，并输出类似下面的信息：


```text
PASS=若干项 WARN=若干项 FAIL=0
```

&emsp;&emsp;这里的具体数量不需要固定，因为每次项目状态不同，检查结果也可能不同。我们重点看的是：脚本是否已经按照前面定义的四类检查组织结果：


```text
目标达成检查
约束遵守检查
质量回归检查
证据充分检查
```

&emsp;&emsp;本次运行中，WARN 主要来自“尚未开始正式迁移”的预期状态，例如：


```text
尚未检测到 LangGraph node / State / Edge
尚未能判断是否存在套壳迁移
CrewAI 仍在运行路径中
live API / SSE 检查暂未开启
部分运行时检查尚未执行
```

&emsp;&emsp;如果这些已经稳定的检查出现 <b>FAIL</b>，就说明当前不应该继续推进，而是必须先修复。这一步完成后，主 Agent 在后续每个 round 内都可以运行：


```text
python3 evals/check_migration_health.py
```

&emsp;&emsp;用于<font color=red>发现问题</font>并自我修复。而在一轮 round 结束、准备交给 verifier 判断前，也必须运行一次最终健康检查，把结果写入 `docs/langgraph_migration_state.md`，作为 verifier 判断是否完成的证据。到这里，3.1 的职责就完成了：我们只生成了自动化健康检查脚本，给后续 loop 提供第一层反馈。至于语义测试怎么固化、每轮状态怎么记录、verifier 怎么判断、主 Agent 怎么遵守循环协议，会放到后面的 3.4.2、3.4.3、3.4.4、3.4.5 和 3.4.6 中继续处理。


#### 3.4.2 建立运行时冒烟检查：服务、API 与 SSE

&emsp;&emsp;静态<font color=red>健康检查</font>能发现很多接口约定问题，比如 API 路由是否还存在、SSE 事件名是否还在、`run_sql` 是否仍然限制只读 SQL。但它不能证明服务真的能启动，也不能证明 `/api/run` 的 SSE 流真的能跑到 `final`。实际迁移中很容易出现这种情况：


```text
静态健康检查 PASS
服务启动失败
或者 /api/run 运行时报错
或者 SSE 只吐出 start/source，随后进入 error
```

&emsp;&emsp;所以还需要一个独立的运行时冒烟测试脚本：


```text
evals/check_runtime_smoke.py
```

&emsp;&emsp;它的职责不是做 benchmark，也不是验证模型输出质量，而是验证“系统运行时关键路径没有断”。本案例里的 runtime smoke test 至少应该覆盖：


```text
脚本能启动服务
/ 返回 200
/api/health 返回 200
/api/agents 返回 workflow/framework/stages 信息
/api/db?table=sales&page=1 返回 200
/api/run?query=...&session=... 的 SSE 能到达 final
final payload 包含 report / chart / sql / columns / rows
如果出现 error 事件，脚本 FAIL 并输出 error payload
```


&emsp;&emsp;这里建议把脚本设计成两种模式：


```text
脚本自己启动服务并检查：
python3 evals/check_runtime_smoke.py --start-server --host 127.0.0.1 --port 8091
```


&emsp;&emsp;这样在人工点击测试时可以先启动服务再运行脚本；在每轮迁移结束时，主 Agent 也可以让脚本自己拉起服务并完成检查。可以继续输入下面的 prompt：


> 💬 输入给 Codex 的提示词

```text
请基于 docs/langgraph_migration_map.md 和 evals/check_migration_health.py，创建运行时冒烟测试脚本。

要求：
1. 不要开始迁移业务代码。
2. 不要替换 CrewAI。
3. 不要修改前端。
4. 不要创建 benchmark。

请创建：

evals/check_runtime_smoke.py

脚本用途：
用于后续每一轮 round 中，在修改了后端运行路径、依赖、API、SSE、数据库访问或 graph/workflow 代码后，验证服务是否真的能启动并跑通关键路径。

脚本只需要支持一种模式：

脚本自己启动服务并检查：

   python3 evals/check_runtime_smoke.py --start-server --host 127.0.0.1 --port 8091

启动服务要求：
- 默认在项目根目录执行；
- 在 --start-server 模式下，脚本必须先确保运行环境，顺序如下：
  1. 如果当前项目根目录的 .venv/bin/python 存在，直接使用它；
  2. 如果当前项目根目录缺少 .venv/bin/python，脚本就在当前项目根目录创建 .venv，并运行：
     .venv/bin/python -m pip install -r requirements.txt
  3. 如果创建环境或安装依赖失败，输出 FAIL；
- .venv 创建和依赖安装都属于 eval 脚本运行准备，不算迁移业务代码；
- .venv 必须保持被 .gitignore 忽略，不要加入 Git。

- 然后脚本必须确保 .env 存在，顺序如下：
  1. 如果当前项目根目录已有 .env，直接使用；
  2. 如果当前项目根目录缺少 .env，脚本输出 FAIL，并提示用户在当前项目根目录手动提供 .env；
  3. 必须确认 .env 被 .gitignore 忽略；
- 不要打印 .env 内容；
- 不要把 .env 加入 Git。


检查项请包括：

1. 服务启动检查
   - 服务是否能在指定 timeout 内启动；
   - /api/health 是否可访问。

2. 基础 API 检查
   - `/` 返回 200；
   - `/api/health` 返回 200，且 JSON 中 `ok` 为 true；
   - `/api/agents` 返回 200，且包含 `pattern`、`framework`、`stages`；
   - `/api/db?table=sales&page=1` 返回 200，且包含 `columns`、`rows`。

3. SSE workflow 检查
   - 请求 `/api/run?query=按品类统计总销售额，取前三名&session=runtime_smoke_test`；
   - SSE 至少应该收到 `start`、`source`、`stage_start`、`sql`、`query_result`、`stage_done`、`final`；
   - 如果收到 `error`，直接 FAIL，并输出 error payload；
   - `final` payload 必须包含：
     - `report`
     - `chart`
     - `sql`
     - `columns`
     - `rows`
   - `chart` 至少包含 `type`、`labels`、`values`；
   - `rows` 应该是数组。

输出要求：
- 每个检查项输出 PASS / WARN / FAIL；
- WARN 不应该让脚本返回非 0；
- FAIL 必须让脚本返回非 0；
- 最后输出 PASS / WARN / FAIL 汇总；
- 输出需要包含：
  - 访问 URL；
  - 是否由脚本启动服务；
  - 服务 PID（如果由脚本启动）；
  - 收到的 SSE event 顺序；
  - final payload 的关键字段检查结果。

完成后请运行：

python3 evals/check_runtime_smoke.py --start-server --host 127.0.0.1 --port 8091

并汇报：
1. 新增文件路径；
2. runtime smoke test 的启动方式；
3. 当前 PASS / WARN / FAIL 汇总；
4. 是否成功启动服务；
5. `/api/run` SSE 是否到达 final；
6. final payload 是否包含 report / chart / sql / columns / rows；
7. 如果有 FAIL，说明失败原因；
8. 是否修改了业务代码、前端或 CrewAI 依赖；
9. git status。
```


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/embedded-cell-304-88524ced.png" width=80%></div>


&emsp;&emsp;这一步完成后，后续主 Agent 在迁移代码之后就不再只依赖静态健康检查。只要本轮触碰了运行路径，就必须把服务真正跑起来，执行：


```text
python3 evals/check_runtime_smoke.py --start-server --host 127.0.0.1 --port 8091
```

&emsp;&emsp;到这里，运行时冒烟检查只证明“关键路径能跑通”。下一小节还要补上真实任务测试集，用来检查“不同输入的关键行为是否仍然正确”。两者一起，才更适合作为提交 verifier 的运行时证据。


#### 3.4.3 建立真实任务测试集：用最小 case 检查语义退化

&emsp;&emsp;运行时冒烟检查能证明服务能启动、<font color=red>API</font> 能访问、SSE 能到达 `final`，但它仍然可能漏掉一个关键问题：<b>流程跑通了，语义却退化了</b>。这在 LLM workflow 迁移里很常见。比如同一条 `/api/run` 能成功返回 `report/chart/sql/columns/rows`，但不同用户问题最后生成了同一条 SQL。此时 smoke test 是 PASS，但业务行为已经不对。所以在 runtime smoke test 之后，应该再加一个很小的<b>真实任务测试集</b>。它不是 benchmark，也不是完整评测集，而是迁移 loop 的语义哨兵：


```text
目标：验证迁移后的 workflow 是否还能根据不同任务输入走出不同、合理的行为路径。
规模：先小后大，通常 3 条简单 case 就够启动。
断言：不要求输出完全一致，只检查关键语义是否命中。
```

&emsp;&emsp;在本案例里，可以让 Codex 生成 3 条简单查询任务：


```text
1. 按品类统计总销售额，取前三名
   期望 SQL 语义：按 category 聚合 amount，返回 Top 3。

2. 按地区统计总销售额，取前三名
   期望 SQL 语义：按 region 聚合 amount，返回 Top 3。

3. 按客户统计总销售额，取前三名
   期望 SQL 语义：按 customer 聚合 amount，返回 Top 3。
```

&emsp;&emsp;推荐把测试集写成独立文件，而不是散落在脚本代码里：


```text
evals/query_smoke_cases.json
```

&emsp;&emsp;建议文件形态如下：


```json
[
  {
    "id": "sales_by_category_top3",
    "query": "按品类统计总销售额，取前三名",
    "expected_sql_terms": ["category", "sum", "amount", "group by", "limit"]
  },
  {
    "id": "sales_by_region_top3",
    "query": "按地区统计总销售额，取前三名",
    "expected_sql_terms": ["region", "sum", "amount", "group by", "limit"]
  },
  {
    "id": "sales_by_customer_top3",
    "query": "按客户统计总销售额，取前三名",
    "expected_sql_terms": ["customer", "sum", "amount", "group by", "limit"]
  }
]
```

&emsp;&emsp;这里的 `expected_sql_terms` 不是要求 SQL 字符串完全一致，而是做最低限度的语义检查。这样不会因为模型把别名写成 `total_sales` 还是 `total_amount` 就误判失败，但能抓住“地区查询却仍然按品类聚合”这类迁移退化。更通用地说，这个小测试集可以按下面的方式提炼：


<div align="center">
<table width="80%">
<thead><tr><th>通用元素</th><th>在任意迁移项目中的含义</th><th>本案例中的例子</th></tr></thead>
<tbody>
<tr><td><code>id</code></td><td>稳定的测试用例标识，便于状态文件和 verifier 引用</td><td><code>sales_by_region_top3</code></td></tr>
<tr><td><code>input</code> / <code>query</code></td><td>用户真实会输入的最小任务</td><td>“按地区统计总销售额，取前三名”</td></tr>
<tr><td><code>expected_terms</code></td><td>对输出做最低限度语义断言</td><td><code>region</code>, <code>sum</code>, <code>amount</code>, <code>group by</code>, <code>limit</code></td></tr>
<tr><td><code>must_reach_final</code></td><td>运行路径必须完整结束</td><td>SSE 到达 <code>final</code></td></tr>
<tr><td><code>required_payload</code></td><td>兼容接口约定必须保留</td><td><code>report</code>, <code>chart</code>, <code>sql</code>, <code>columns</code>, <code>rows</code></td></tr>
<tr><td><code>non_goal</code></td><td>明确不是性能评测或严格黄金答案匹配</td><td>不比较耗时，不要求 SQL 字符串完全一致</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;可以继续输入下面的 prompt：


> 💬 输入给 Codex 的提示词

```text
请基于 evals/check_runtime_smoke.py，新增一个真实任务测试集，并接入 runtime smoke。

要求：
1. 不要开始迁移业务代码。
2. 不要替换 CrewAI。
3. 不要修改前端。
4. 不要创建 benchmark。

请创建：

evals/query_smoke_cases.json

测试集包含 3 条简单查询：

1. sales_by_category_top3
   - query: 按品类统计总销售额，取前三名
   - expected_sql_terms: category, sum, amount, group by, limit

2. sales_by_region_top3
   - query: 按地区统计总销售额，取前三名
   - expected_sql_terms: region, sum, amount, group by, limit

3. sales_by_customer_top3
   - query: 按客户统计总销售额，取前三名
   - expected_sql_terms: customer, sum, amount, group by, limit

请更新 evals/check_runtime_smoke.py：
- 脚本读取 evals/query_smoke_cases.json；
- 逐条请求 `/api/run`；
- 每条 case 都必须到达 `final`；
- 每条 final payload 都必须包含 `report`、`chart`、`sql`、`columns`、`rows`；
- 对 final payload 中的 `sql` 做最低限度语义检查；
- 不要求 SQL 完全一致；
- 不做 benchmark；
- 不比较耗时；
- 如果某条 case 没到 `final`、收到 `error`、payload 缺字段或 SQL 缺少期望语义词，输出 FAIL 并让脚本非 0 退出。

完成后请运行：

python3 evals/check_runtime_smoke.py --start-server --host 127.0.0.1 --port 8091

并汇报：
1. 新增文件路径；
2. 3 条真实任务测试集是否逐条通过；
3. 每条测试生成的 SQL 是否满足最低语义检查；
4. 如果有 FAIL，说明失败原因；
5. 是否修改了业务代码、前端或 CrewAI 依赖；
6. git status。
```


&emsp;&emsp;<b>小结：真实任务测试集的通用作用</b>

<div align="center">
<table width="80%">
<thead><tr><th>检查层</th><th>能证明什么</th><th>不能证明什么</th><th>应放在哪个阶段</th></tr></thead>
<tbody>
<tr><td>静态健康检查</td><td>文件、路由、接口约定、边界没有明显破坏</td><td>服务是否真的能跑</td><td>迁移 round 内和结束前</td></tr>
<tr><td>运行时冒烟检查</td><td>服务能启动，关键 API/SSE 能到 final</td><td>不同任务语义是否仍然正确</td><td>修改运行路径后</td></tr>
<tr><td>真实任务测试集</td><td>不同输入能产生符合预期的关键行为</td><td>全量业务正确性或性能表现</td><td>runtime smoke 之后、verifier 之前</td></tr>
<tr><td>verifier 复核</td><td>证据是否足以判定 complete/fix_required</td><td>自动修复代码</td><td>每轮 round 结束</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;这一层检查的价值在于：它把“能跑”推进到“跑得像原系统”。对于迁移任务来说，这通常比只看一个 happy path smoke 更可靠。


#### 3.4.4 建立迁移状态记录：用状态文件承接每轮证据

&emsp;&emsp;健康检查脚本可以告诉我们“一次 round 的检查结果是什么”，但 loop 跑多轮以后，仅靠终端输出不够。如果结果只留在聊天记录里，Agent 很容易在后面几轮忘记前面的判断，或者重复分析同一个问题。所以需要一个状态文件：


```text
docs/langgraph_migration_state.md
```

&emsp;&emsp;它相当于整个迁移 loop 的外部记忆，用来记录当前轮次、本轮目标、修改文件、健康检查结果、runtime smoke test 结果、verifier 判断和下一步计划。可以继续输入：


```text
请基于 docs/langgraph_migration_map.md、evals/check_migration_health.py、evals/check_runtime_smoke.py 和 evals/query_smoke_cases.json，创建中文迁移 loop 的状态文件模板。

要求：
1. 不要开始迁移业务代码。
2. 不要替换 CrewAI。
3. 不要修改前端。

请创建：

docs/langgraph_migration_state.md

这个文件用于记录后续每一轮 round 的状态。

请包含以下内容：

1. Loop 配置
   - max_rounds：先设置为 6
   - current_round：0
   - status：in_progress

2. Round 记录说明
   - 说明每个 round 需要记录开始状态和结束状态；

3. 总目标
   - 将 CrewAI 编排迁移为 LangGraph；
   - 保持 API / SSE / DB / Frontend 契约兼容；
   - 避免把旧 run_events() 套壳成单个 LangGraph node。

4. 完成条件
   - LangGraph 真实表达旧 workflow 的主要阶段；
   - 存在多个业务 node；
   - SQL retry 使用 conditional edge；
   - CrewAI 不再处于运行路径；
   - 健康检查无 FAIL；
   - 如果本轮修改运行路径，runtime smoke test 无 FAIL，且 /api/run SSE 到达 final；
   - 如果本轮修改运行路径，真实任务测试集无 FAIL，且关键输出语义没有明显跑偏；
   - API / SSE / DB / Frontend 契约兼容；
   - 关键 WARN 已处理或有明确说明；
   - verifier 返回 complete。

5. 当前健康检查基线
   请记录当前 evals/check_migration_health.py 的检查结果摘要。
   如果还没有运行，请先运行：
   python3 evals/check_migration_health.py

   按四类记录：
   - 目标达成检查：当前 PASS / WARN / FAIL 摘要；
   - 约束遵守检查：当前 PASS / WARN / FAIL 摘要；
   - 质量回归检查：当前 PASS / WARN / FAIL 摘要；
   - 证据充分检查：当前 PASS / WARN / FAIL 摘要。

6. 每轮记录模板
   每个 round 至少需要记录：

   Round 开始记录：
   - round number；
   - round goal；
   - planned scope；
   - expected checks；
   - known risks。

   Round 结束记录：
   - changed files；
   - final health check summary；
   - runtime smoke test summary（如果本轮修改了运行路径、API、SSE、依赖或 graph/workflow 代码）；
   - real task smoke test summary（如果本轮修改了运行路径、API、SSE、依赖或 graph/workflow 代码）；
   - target achievement summary；
   - constraint compliance summary；
   - quality regression summary；
   - evidence sufficiency summary；
   - remaining issues；
   - verifier decision；
   - next action。

7. 当前状态
   由于现在还没有开始正式迁移，请把 current_round 设置为 0。
   请记录当前只有 baseline、migration_map、health check、runtime smoke 和真实任务测试集准备工作。
   如果当前健康检查存在 WARN，请标记为“expected before migration”，不要当作阻塞项。

完成后请汇报：
- 新增文件路径；
- max_rounds 当前设置；
- round 记录规则；
- 当前健康检查基线摘要；
- 状态文件包含哪些章节；
- git status。
```


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/embedded-cell-329-46665ebd.png" width=80%></div>


&emsp;&emsp;Codex 执行完成后，创建了迁移状态文件模板：


```text
docs/langgraph_migration_state.md
```

&emsp;&emsp;当前状态文件里的 loop 配置是：


```text
max_rounds: 6
current_round: 0
status: in_progress
```

&emsp;&emsp;这表示迁移循环还没有正式开始，目前处在第 0 轮准备阶段。同时，Codex 重新运行了一次健康检查，并把当前结果作为基线写入状态文件：


```text
python3 evals/check_migration_health.py
PASS=14 WARN=7 FAIL=0
```

&emsp;&emsp;这组数字本身不是重点；后续项目状态变化后结果也会变化。重点是状态文件已经开始记录目标达成、约束遵守、质量回归和证据充分这四类检查基线。状态文件主要包含这些章节：


```text
Loop 配置
Round 记录说明
总目标
完成条件
当前健康检查基线
每轮记录模板
当前状态 / Round 0
状态更新规则
```

&emsp;&emsp;这一步把迁移过程从聊天记录中移出来，变成主 Agent 和 verifier 都能读取的外部状态文件。


#### 3.4.5 配置只读复核者：两态 verifier

&emsp;&emsp;健康检查脚本负责输出 `PASS / WARN / FAIL`，状态文件负责记录过程。但一轮 round 结束时，还需要独立角色判断：本轮是否真的推进了迁移目标、是否突破边界、是否满足最终完成条件。因此这里引入 <b>maker-checker 分离</b>：


```text
主 Agent：负责执行迁移、运行健康检查、自我修复、更新状态文件
verifier subagent：负责在一轮 round 结束后，只读审查证据，并判断是否完成
```

&emsp;&emsp;verifier 不是最后才临时出现一次，而是在正式迁移循环开始前就提前配置好。它不写代码，只根据证据判断当前 loop 状态。每轮 round 结束时主要读取这些证据：


```text
docs/langgraph_migration_map.md
docs/langgraph_migration_state.md
python3 evals/check_migration_health.py 的最终输出
服务启动与 runtime smoke test 输出（本轮涉及运行路径、API、SSE、依赖或 graph 代码时必须提供）
真实任务测试集输出（本轮涉及运行路径、API、SSE、依赖或 graph 代码时必须提供）
本轮 git diff 摘要
主 Agent 本轮目标和结果说明
```

&emsp;&emsp;然后只输出两种核心 decision：


```text
complete       总迁移目标已经满足，可以停止 loop
fix_required   本轮仍有问题，需要进入下一轮继续修复
```

&emsp;&emsp;主 Agent 每一轮都是围绕总目标完整尝试一次迁移。只要 verifier 不认为目标完成，就进入下一轮。项目级 subagent 可以放在：


```text
agents/langgraph-migration-verifier.toml
```


&emsp;&emsp;可以继续输入下面的提示词：


> 💬 输入给 Codex 的提示词

```text
请基于 docs/langgraph_migration_map.md、docs/langgraph_migration_state.md 和 evals/check_migration_health.py，用中文配置一个项目级 verifier subagent。

要求：
1. 不要开始迁移业务代码。
2. 不要替换 CrewAI。
3. 不要修改前端。

请创建：

agents/langgraph-migration-verifier.toml

这个 subagent 只负责只读复核，不负责写代码，不负责迁移实现。

模型配置：
- model = "gpt-5.4"
- model_reasoning_effort = "high"
- sandbox_mode = "read-only"

它在每一轮 round 结束后，根据以下证据判断本轮是否已经满足最终完成条件：
- docs/langgraph_migration_map.md；
- docs/langgraph_migration_state.md；
- python3 evals/check_migration_health.py 的最终输出；
- 如果本轮修改了后端运行路径、API、SSE、依赖或 graph 代码，需要提供服务启动与 runtime smoke test 结果，包括启动命令、访问 URL、/api/health、关键 API 以及 /api/run SSE 是否到达 final；
- 如果本轮修改了后端运行路径、API、SSE、依赖或 graph 代码，需要提供真实任务测试集结果，包括每条 case 是否到达 final、final payload 是否完整、关键输出语义是否命中；
- 本轮 git diff 摘要；
- 主 Agent 本轮目标和结果说明。

它只需要输出两种核心 decision：

1. complete
   总迁移目标已经满足，可以停止 loop。

2. fix_required
   本轮仍有问题，不能停止，需要进入下一轮继续修复。

不要设计第三种中间 decision。只要最终完成条件没有全部满足，就统一返回 fix_required，并说明下一轮应修复什么。

请在 TOML 中写清楚 verifier 的判断标准，并按照四类组织：

1. 目标达成检查
   - 本轮是否真的推进了 LangGraph node / State / Edge 迁移；
   - 是否避免把旧 run_events() 套成一个大 node；
   - 是否已经达到 docs/langgraph_migration_state.md 中的完成条件。

2. 约束遵守检查
   - 是否突破迁移边界；
   - 是否破坏 API / SSE / DB / Frontend 兼容性；
   - 是否修改了原则上不应该修改的文件。

3. 质量回归检查
   - SQL retry、chart fallback、历史上下文、只读 SQL 安全边界是否仍然保留；
   - 是否有证据表明旧 workflow 的关键行为没有下降。

4. 证据充分检查
   - 最终健康检查是否有 FAIL；
   - WARN 是否合理；
   - 如果本轮修改了运行路径，是否已经启动服务并完成 runtime smoke test；
   - 如果本轮修改了运行路径，是否已经运行真实任务测试集，且没有出现关键语义跑偏；
   - 主 Agent 是否提供了本轮目标、diff 摘要、最终健康检查结果和状态文件更新；
   - 是否达到或超过 max_rounds。

decision 规则：

- 如果存在 FAIL，返回 fix_required。
- 如果本轮突破迁移边界，返回 fix_required。
- 如果证据不足，返回 fix_required，并说明缺少什么证据。
- 如果本轮修改了运行路径但没有服务启动证据、关键 API smoke 失败，或 /api/run SSE 没有到达 final，返回 fix_required。
- 如果本轮修改了运行路径但没有真实任务测试集证据，或测试集显示关键语义跑偏，返回 fix_required。
- 如果 WARN 对应的是完成条件中的关键项，且没有明确理由保留，返回 fix_required。
- 只有在 docs/langgraph_migration_state.md 中的完成条件全部满足时，才能返回 complete。
- 如果已经达到 max_rounds 仍未满足完成条件，返回 fix_required，并要求主 Agent 停止并汇报阻塞点，而不是继续无限循环。

完成后请汇报：
- 新增文件路径；
- verifier agent 名称；
- 使用的模型；
- reasoning effort；
- sandbox 模式；
- 它的 decision 类型；
- 它依赖哪些证据；
- git status。
```


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch03-migration-original-10-800e9f29.png" width=80%></div>

&emsp;&emsp;Codex 执行完成后，创建了项目级 verifier subagent 配置：


```text
agents/langgraph-migration-verifier.toml
```


&emsp;&emsp;这个 verifier 的基本配置是：


```text
verifier agent 名称：langgraph-migration-verifier
使用模型：gpt-5.4
reasoning effort：high
sandbox：read-only
```

&emsp;&emsp;它只做只读复核，不修改代码，也不负责迁移实现。它的核心 decision 是：


```text
complete
fix_required
```

&emsp;&emsp;在这个案例里，判断逻辑保持两态：<b>complete 就停止，不 complete 就返回 fix_required 并进入下一轮修复</b>。它每轮依赖的证据包括：


```text
docs/langgraph_migration_map.md
docs/langgraph_migration_state.md
python3 evals/check_migration_health.py 最终输出
本轮 git diff 摘要
主 Agent 本轮目标和结果说明
```

&emsp;&emsp;当前 `git status` 显示：


```text
## migration/langgraph-loop
?? agents/
?? docs/langgraph_migration_state.md
?? evals/
```


&emsp;&emsp;展开后可以看到新增的 verifier 配置文件：


```text
?? agents/langgraph-migration-verifier.toml
```


&emsp;&emsp;这说明当前只新增了 loop engineering 的复核配置，没有开始业务迁移，没有替换 <font color=red>CrewAI</font>，也没有修改前端。这里要注意，TOML 文件只是定义了 verifier subagent。它不会天然保证每一轮 round 结束后自动触发。要想让它固定参与后续 loop，还需要在下一步把“每轮 round 结束后必须调用 verifier，并把最终健康检查结果、diff 摘要、状态文件更新交给它判断”写进 `AGENTS.md`。这也说明 loop engineering 不能只靠一个 prompt，而是要把流程规则、状态文件、检查脚本和 verifier 组合起来。


#### 3.4.6 固化主循环协议：AGENTS.md 与提交边界

&emsp;&emsp;健康检查脚本、状态文件和 verifier 还不能保证主 Agent 每轮都按流程执行，所以要把主 loop 协议写进项目根目录的 `AGENTS.md`。它不是硬执行器，而是项目级操作协议，用来固定 round 流程、证据展示、停止条件和边界保护。可以继续输入下面的 prompt：


> 💬 输入给 Codex 的提示词

```text
请把后续主 Agent 执行迁移 loop 的规则用中文写入项目根目录 AGENTS.md。

要求：
1. 不要开始迁移业务代码。
2. 不要替换 CrewAI。
3. 不要修改前端。
4. 如果项目已经存在 AGENTS.md，请只追加一个“LangGraph Migration Loop Rules”小节，不要覆盖原有内容。
5. 如果项目不存在 AGENTS.md，请创建它。

请在 AGENTS.md 中写入以下规则：

一、round 定义

这里的 round 不是改一个文件，也不是做一个小功能。

一个 round 指主 Agent 围绕总迁移目标完成一次完整迁移尝试。  
在一个 round 内部，主 Agent 可以读取代码、修改代码、运行健康检查、根据错误自行修复，并多次重复这些动作。

一个 round 结束前，必须运行一次最终健康检查。凡本轮修改了后端运行路径、依赖、API、SSE、数据库访问或 graph/workflow 代码，还必须启动服务并运行 runtime smoke test。随后更新状态文件，并调用 verifier subagent 判断是否完成。

二、每轮 round 固定流程

每一轮 round 都必须按下面顺序执行：

1. 读取 docs/langgraph_migration_map.md，确认旧 workflow、候选设计、迁移边界和健康检查项；
2. 读取 docs/langgraph_migration_state.md，确认 current_round、max_rounds、remaining issues 和上一轮 verifier decision；
3. 围绕总迁移目标进行一次完整迁移尝试；
4. 在本轮内部可以多次运行健康检查：

   python3 evals/check_migration_health.py

5. 根据健康检查结果，在本轮内部自行修复明显问题；
6. 在本轮结束前，必须再次运行最终健康检查；
7. 如果本轮修改了后端运行路径、依赖、API、SSE、数据库访问或 graph/workflow 代码，必须启动服务并运行最小 runtime smoke test：

   - 如迁移 worktree 缺少 .env，请停止并提示用户在当前项目根目录提供 .env；
   - 如缺少虚拟环境或依赖，可在迁移 worktree 内创建 .venv 并安装 requirements.txt；
   - 启动服务，例如：

     .venv/bin/python -m uvicorn backend.app:app --host 127.0.0.1 --port 8091

   - 至少检查：
     - 首页 `/` 返回 200；
     - `/api/health` 返回 200；
     - `/api/agents` 返回迁移后的 framework / stages 信息；
     - `/api/db?table=sales&page=1` 返回 200；
     - `/api/run?query=...&session=...` 的 SSE 能到达 `final`，且 final payload 包含 `report`、`chart`、`sql`、`columns`、`rows`；
     - `evals/query_smoke_cases.json` 中的 3 条真实任务测试集逐条到达 `final`；
     - 每条测试生成的 SQL 至少包含该 case 对应的关键字段和聚合语义，例如 category / region / customer、sum、amount、group by、limit；
   - 如果服务无法启动、关键 API 不通或 SSE 返回 `error`，必须在本轮内修复，不能把该轮提交给 verifier 判 complete；
   - 如果真实任务测试集没有逐条通过，或 SQL 语义明显跑偏，必须在本轮内修复，不能只因为单条 smoke 到达 `final` 就提交 verifier 判 complete；
   - 测试结束后说明服务是否仍保持运行，或者是否已停止。

8. 根据最终健康检查和 runtime smoke test 结果，更新 docs/langgraph_migration_state.md；
9. 汇报本轮证据，包括：
   - 本轮目标；
   - 修改文件；
   - 最终健康检查 PASS / WARN / FAIL；
   - runtime smoke test 结果（如本轮适用）；
   - git diff 摘要；
   - remaining issues；
   - next action；
10. 调用 langgraph-migration-verifier subagent 做只读复核；
11. 根据 verifier decision 决定下一步：
   - complete：停止 loop，准备最终汇报；
   - fix_required：进入下一轮 round 继续修复。

三、最大轮数

- 默认 max_rounds 为 6；
- 每轮开始前检查 current_round；
- 如果达到 max_rounds 仍未 complete，必须停止并汇报阻塞点；
- 不要在没有人工确认的情况下无限继续。

四、每轮必须展示的证据

每一轮 round 结束时，主 Agent 必须展示：

- 本轮目标；
- 修改文件列表；
- 最终健康检查结果；
- runtime smoke test 结果（如果本轮修改了运行路径/API/SSE/依赖/graph）；
- 真实任务测试集结果（如果本轮修改了运行路径/API/SSE/依赖/graph）；
- git diff 摘要；
- docs/langgraph_migration_state.md 的更新摘要；
- verifier decision；
- 下一轮计划或最终汇报。

五、停止条件

只有满足以下情况之一，才能停止 loop：

1. verifier 返回 complete；
2. 达到 max_rounds，需要停止并汇报阻塞点；
3. 出现无法自动解决的阻塞，需要人工确认。

如果 verifier 返回 fix_required，不能停止，必须进入下一轮 round 或请求人工确认。

六、边界规则

迁移过程中必须遵守 docs/langgraph_migration_map.md 中的边界要求：

- 不要随意修改 frontend/index.html；
- 不要随意修改 backend/tools.py、backend/runtime.py、backend/store.py、backend/seed.py、backend/config.py、backend/app.py；
- 不要破坏 API 路径、请求参数、响应字段；
- 不要破坏 SSE 事件名、事件顺序和 payload 字段；
- 不要破坏数据库 schema 和只读 SQL 安全边界；
- 如果必须修改边界外文件，先说明原因、影响范围和兼容策略，再等待确认。

七、提交规则

- 每完成一个明确阶段，应使用 Git 固化状态；
- 迁移地图、反馈机制、每轮迁移代码应尽量分开提交；
- 如果 git diff 中出现无关文件或边界外文件，必须说明原因。

完成后请汇报：
- 是否创建或更新 AGENTS.md；
- 追加了哪些主要规则；
- 是否修改了业务代码、前端或 CrewAI 依赖；
- git status。
```


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch03-migration-original-11-a7aa217f.png" width=80%></div>

&emsp;&emsp;Codex 执行完成后，创建了项目根目录的 `AGENTS.md`。项目此前不存在这个文件，所以这次是新建，而不是追加。`AGENTS.md` 已经写入 round 流程、最大轮数、证据展示、停止条件、迁移边界和 Git 提交规则。当前 `git status` 显示：


```text
## migration/langgraph-loop
?? agents/
?? AGENTS.md
?? docs/langgraph_migration_state.md
?? evals/
```


&emsp;&emsp;这说明当前新增的都是 loop engineering 相关的工程辅助文件，没有修改业务代码、前端、<font color=red>CrewAI</font> 依赖或项目运行逻辑。到这里，主 Agent 的 loop 协议已经被写入 `AGENTS.md`，第三步的反馈机制完整了。完成 `AGENTS.md` 之后，进入正式迁移前还应该再做一件事：<b>把第三步新增的反馈机制文件单独提交一次</b>。因为当前这些文件还都处于未跟踪状态：


```text
agents/langgraph-migration-verifier.toml
AGENTS.md
docs/langgraph_migration_state.md
evals/check_migration_health.py
evals/check_runtime_smoke.py
evals/query_smoke_cases.json
```


&emsp;&emsp;这些文件属于 loop engineering 基础设施，不是业务迁移代码，应该和后续正式迁移代码分开提交。可以继续输入下面的提示词：


> 💬 输入给 Codex 的提示词

```text
请只提交第三步新增或更新的 loop feedback 工程文件，不要修改任何业务代码。

当前预期新增或更新的文件包括：
- evals/check_migration_health.py
- evals/check_runtime_smoke.py
- evals/query_smoke_cases.json
- docs/langgraph_migration_state.md
- agents/langgraph-migration-verifier.toml
- AGENTS.md

请执行：
1. git status --short
2. 确认未提交改动只包含上述反馈机制文件；其中 evals/check_runtime_smoke.py 可能已经在真实任务测试集步骤中被更新
3. 不要把 .env、.venv/、evals/results/、worktrees/、业务代码或前端文件加入提交
4. git add evals/check_migration_health.py evals/check_runtime_smoke.py evals/query_smoke_cases.json docs/langgraph_migration_state.md agents/langgraph-migration-verifier.toml AGENTS.md
5. git commit -m "chore: add migration loop feedback infrastructure"
6. 提交后汇报：
   - commit hash
   - git status --short --branch

如果发现 docs/langgraph_migration_map.md、业务代码、前端文件、CrewAI 依赖、.env、.venv/、evals/results/、worktrees/ 或其他非预期文件出现在未提交改动里，请停止并说明，不要提交。
```


&emsp;&emsp;Codex 提交完成后，正式迁移前的准备工作就完成了。下一步进入第四小节：执行迁移 round。每一轮都是主 Agent 围绕总迁移目标的一次完整尝试；round 结束前必须运行最终健康检查。凡本轮修改运行路径，还必须运行 runtime smoke test 和真实任务测试集。随后更新状态文件，并交给 verifier 判断是 `complete` 还是 `fix_required`。


### 3.5 执行迁移循环：让 Agent 按协议自主推进

&emsp;&emsp;前面已经把迁移地图、状态文件、健康检查、真实任务测试集、verifier 和 `AGENTS.md` 都准备好了。现在就可以正式让 <font color=red>Codex</font> 按照这些规则开始迁移。这里不需要再手动指定“先改哪个函数、再改哪个类”。因为前面已经通过迁移地图、状态文件、健康检查脚本、真实任务测试集和 `AGENTS.md` 约束好了目标、边界、检查方式和停止条件。所以这一节的关键，是把任务交给主 Agent，让它按照已有协议自主推进：


```text
读取 AGENTS.md
读取 migration_map
读取 migration_state
确认当前轮次和剩余问题
围绕总目标执行迁移
过程中运行健康检查并自行修复
round 结束前运行最终健康检查
如果本轮修改运行路径/API/SSE/依赖/graph，则启动服务并跑 runtime smoke test
同时跑真实任务测试集，确认不同查询能生成对应 SQL 语义
更新状态文件
调用 verifier 判断 complete / fix_required
```

&emsp;&emsp;这一轮迁移的总目标仍然是：


```text
从 CrewAI 编排迁移到 LangGraph
保持 API / SSE / DB / Frontend 契约兼容
避免 run_events() 套壳
保留 SQL retry、chart fallback、历史上下文等旧行为
```

&emsp;&emsp;我们已经搭好了 loop engineering 机制，接下来驱动 Agent 按规则自己推进迁移</b>。可以输入下面的提示词：


> 💬 输入给 Codex 的提示词

```text
请按照项目中已经建立好的 LangGraph migration loop 协议，持续推进迁移。
确认自己在项目内 `worktrees/langgraph` 迁移工作区，也就是当前迁移 worktree 项目根目录，并且处于 migration/langgraph-loop 分支。

开始前请读取并遵守：

- AGENTS.md
- docs/langgraph_migration_map.md
- docs/langgraph_migration_state.md
- evals/check_migration_health.py
- evals/check_runtime_smoke.py
- agents/langgraph-migration-verifier.toml

请根据 docs/langgraph_migration_state.md 判断当前 round、max_rounds、remaining issues、上一轮健康检查结果和上一轮 verifier decision，并决定本轮是迁移、修复还是收尾。

要求：

1. 本任务是持续 loop，不是单轮任务；
2. 每轮开始前更新 docs/langgraph_migration_state.md；
3. 每轮修改范围尽量小；
4. 每轮结束前必须运行：
   python3 evals/check_migration_health.py
5. 如果本轮修改了后端运行路径、依赖、API、SSE、数据库访问或 graph/workflow 代码，每轮结束前必须自己启动服务并运行 runtime smoke test：

   python3 evals/check_runtime_smoke.py --start-server --host 127.0.0.1 --port 8091
6. 如果本轮修改了后端运行路径、依赖、API、SSE、数据库访问或 graph/workflow 代码，每轮结束前还必须运行真实任务测试集：
   - 测试集来自 `evals/query_smoke_cases.json`；
   - 至少包含 3 条简单查询：按品类、按地区、按客户统计总销售额，取前三名；
   - 每条查询都必须到达 `final`；
   - 每条 final payload 都必须包含 `report`、`chart`、`sql`、`columns`、`rows`；
   - 每条生成 SQL 都必须满足该 case 的最低语义检查，例如 category / region / customer、sum、amount、group by、limit；
   - 如果真实任务测试集失败，必须进入修复，不允许仅凭单条 runtime smoke 通过就判定完成；
7. 每轮结束后必须根据 agents/langgraph-migration-verifier.toml 的规则，调用 langgraph-migration-verifier 做只读复核；
8. 如果 verifier decision 是 fix_required，立即进入下一轮，优先修复 remaining issues，不要等待我继续；
9. 只有以下情况才允许停止并最终汇报：
   - verifier decision 是 complete；
   - 达到 max_rounds；
   - 连续多轮卡在同一问题，且能明确说明无法自动解决的原因。

允许替换 CrewAI 相关内容；如果与旧文件存在冲突，以本次 prompt 为准。
```


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/ch03-migration-original-12-90ee8fb4.png" width=80%></div>

&emsp;&emsp;从执行结果可以看到，<font color=red>Codex</font> 并不是简单执行一次代码修改就结束，而是按照我们前面定义好的 loop 协议继续推进。这一次迁移过程中，Codex 先完成了第一轮代码迁移，把原来的 CrewAI 编排逻辑改成了 LangGraph 流程，并运行健康检查。但在第一轮结束时，verifier 并没有直接判定完成，而是返回了 `fix_required`。代码迁移虽然已经完成，但本轮用于判定完成的证据还不完整，例如迁移后状态记录、runtime smoke 或真实任务测试集结果还没补齐。按照我们前面定义的规则，`fix_required` 不是停止条件，所以 Codex 没有等待我们继续输入，而是自动进入下一轮。


&emsp;&emsp;这也是 Loop Engineering 的核心价值：人不再负责一条条追问“继续修一下”“再检查一下”“文档也补一下”，而是提前设计好循环协议，让 Agent 按照规则自己推进，直到满足明确的完成条件。


&emsp;&emsp;然后我们可以让它把服务启动起来看一下。


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/embedded-cell-386-d0201e94.png" width=80%></div>


&emsp;&emsp;可以看到服务可以正常运行。


> <b>提示</b>: 后续轮次停止的锚点很明确。第一轮 `fix_required` 是因为证据不完整；下一轮需要补齐状态记录、runtime smoke、真实任务测试集等证据。只有健康检查无 FAIL、必要 smoke 到达 `final`、真实任务 case 不跑偏，并且 verifier 返回 `complete` 时，loop 才能停止。

### 3.6 小结：把迁移过程沉淀成 Loop Engineering Memory

&emsp;&emsp;这一节不是继续<font color=red>迁移</font>代码，而是回看前面每一步：哪些东西只是本案例的临时细节，哪些东西值得沉淀成下一次迁移还能复用的 memory。Loop Engineering 里的 memory 不是简单保存聊天记录。更有价值的是把循环中学到的事实、边界、检查项和判断规则写到外部文件里，让后续 round、下一个项目、甚至另一个 Agent 都能直接读取和复用。先把前文每个小节和可沉淀的 memory 对齐：


<div align="center">
<table width="80%">
<thead><tr><th>前文小节</th><th>本节完成的事情</th><th>可沉淀的 memory</th><th>下次迁移可复用什么</th></tr></thead>
<tbody>
<tr><td>3.1 方法总览</td><td>说明为什么不用一次性 prompt，而要手动构建 loop</td><td>loop engineering 的基本构成</td><td>先外部化目标、边界、检查、状态和复核</td></tr>
<tr><td>3.2 隔离实验环境</td><td>准备 Git baseline 和迁移 worktree</td><td>环境隔离 memory</td><td>任何迁移先保留 baseline，再开独立 worktree</td></tr>
<tr><td>3.3.1 识别真实运行流程</td><td>只读分析旧系统入口和执行顺序</td><td>旧系统事实 memory</td><td>先问“系统真实怎么跑”，不要直接按框架名迁移</td></tr>
<tr><td>3.3.2 设计候选目标结构</td><td>把旧行为映射为目标架构里的阶段</td><td>候选设计 memory</td><td>迁移目标要来自旧行为阶段，而不是 API 机械替换</td></tr>
<tr><td>3.3.3 识别迁移边界</td><td>明确哪些文件和接口约定不能破坏</td><td>边界 memory</td><td>API、事件、DB、前端、历史数据要先列成保护清单</td></tr>
<tr><td>3.3.4 设计反馈检查项</td><td>列出每轮后要检查什么</td><td>检查思路 memory</td><td>检查要覆盖目标达成、约束遵守、质量回归、证据充分</td></tr>
<tr><td>3.3.5 形成迁移地图</td><td>把前面分析写成文档</td><td>项目事实 memory</td><td>让后续 Agent 先读 map，再动代码</td></tr>
<tr><td>3.4.1 静态健康检查</td><td>把检查项落成脚本</td><td>静态检查 memory</td><td>用脚本把“有没有破坏边界”变成 PASS/WARN/FAIL</td></tr>
<tr><td>3.4.2 运行时冒烟检查</td><td>验证服务启动和关键路径</td><td>运行验证 memory</td><td>只要改运行路径，就必须证明服务真的能跑到 final</td></tr>
<tr><td>3.4.3 真实任务测试集</td><td>用少量真实 case 检查语义退化</td><td>任务样本 memory</td><td>不能只看跑通，还要看不同输入是否触发正确行为</td></tr>
<tr><td>3.4.4 迁移状态记录</td><td>记录 round、检查结果和剩余问题</td><td>过程状态 memory</td><td>多轮循环不能只靠聊天记忆，要写状态文件</td></tr>
<tr><td>3.4.5 只读复核者</td><td>配置 verifier 判断 complete / fix_required</td><td>复核规则 memory</td><td>主 Agent 不自己宣布完成，要交给只读角色复核</td></tr>
<tr><td>3.4.6 主循环协议</td><td>固化每轮执行顺序和停止条件</td><td>操作协议 memory</td><td>把读资料、修改、检查、修复、复核写成项目规则</td></tr>
<tr><td>3.5 执行迁移循环</td><td>让 Agent 按协议持续推进</td><td>round 经验 memory</td><td><code>fix_required</code> 不是失败，而是下一轮输入</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;这张表的重点是：不要把所有细节都沉淀成 skill。比如项目路径、销售表字段、具体端口、这一次的 round 结果，都只是本案例数据。真正值得沉淀的是可迁移的判断方式：


```text
先固定基线
先只读理解旧系统
先写迁移地图
把检查项变成脚本
把真实任务变成小测试集
把每轮状态写进文件
用 verifier 做两态复核
没有 complete 就继续下一轮
```

&emsp;&emsp;因此，第 5 节的沉淀动作不应该一上来就让 Codex 生成一个很重的完整 skill。更稳的做法是让 Codex 直接生成一份 `memory.md` 风格的经验沉淀文件：区分“本案例事实”和“可复用规则”，并在文件最后判断这些经验是否已经适合升级成 skill。可以继续输入下面的 prompt，让 Codex 一次性完成经验沉淀：


> 💬 输入给 Codex 的提示词

```text
请基于当前项目中的迁移 loop 产物，直接创建一份 Loop Engineering memory 文件。

目标：
把这次 CrewAI 到 LangGraph 迁移过程中可复用的经验沉淀出来。
注意：不要写业务迁移代码，不要再次执行迁移 round。
请先区分哪些是“本案例事实”，哪些是“可复用迁移规则”。

请只读参考以下材料：
- docs/langgraph_migration_map.md；
- docs/langgraph_migration_state.md；
- evals/check_migration_health.py；
- evals/check_runtime_smoke.py；
- evals/query_smoke_cases.json；
- agents/langgraph-migration-verifier.toml；
- AGENTS.md。

请创建或更新：

docs/loop_engineering_memory.md

文件内容不要超过一页，包含：

1. Overview
   - 用 3-5 句话说明这次迁移沉淀出的核心方法。

2. Reusable Rules
   - 用表格列出哪些经验可以复用到其他迁移项目。

3. Case-Specific Details
   - 用表格列出哪些内容只是本案例细节，不应该写死到通用方法里。

4. Minimal Loop Checklist
   - 用一张短表列出下次迁移最少要保留哪些步骤。

5. Skill Readiness
   - 用 3-5 条判断：这些经验现在是否适合升级成通用 Codex skill，还是应该先作为项目 memory 保留。
   - 如果适合升级成 skill，请在本文件最后追加一个 “Candidate Skill Scope” 小节，列出 skill 应该包含和不应该包含的内容。

要求：
- 不要输出过长模板；
- 不要把 CrewAI、LangGraph、销售报表字段写成通用规则；
- 重点沉淀判断方法和流程，而不是复述本项目所有细节；
- 不要创建完整 skill；
- 最后汇报新增或更新的文件路径、主要章节、是否建议后续升级成 skill、git status。
```


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/embedded-cell-396-d2aab14d.png" width=80%></div>


&emsp;&emsp;可以看到生成了一个沉淀经验的memory.md，后续如果我们要做框架迁移，可以使用这个memory，来参考本次迁移的经验，这就loop engineering中外循环的重要部分。


## <center>第四章：把 Loop Engineering 变成可复用工程习惯</center>

&emsp;&emsp;前三章已经走过三种视角：第一章给出 Prompt、Context、Harness、Loop 的责任边界；第二章用每日 AI 新闻日报展示 Codex 里的 Automation、Skill、Goal、state、memory 和 verifier 怎样组合成自动化循环；第三章在框架迁移里展示没有内置 Goal 时，也能用 baseline、worktree、迁移地图、检查脚本、状态文件和只读复核者搭出透明 loop。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/loop-engineering/2026-07-09/embedded-cell-400-e338ac32.png" width=80%></div>


&emsp;&emsp;这一章把这些经验压成一套开工顺序。下次拿到一个真实任务，我们先判断它是否值得进入 loop，再选择最小部件组合，接着写清第一轮运行协议；跑偏时按根因修 loop，最后用停止条件和人工门决定继续、重跑、交给人还是收束。

### 4.1 先判任务形态

&emsp;&emsp;Loop Engineering 的第一步不是选工具，而是判断任务形态。很多任务只需要一次清晰输入，不需要 state、verifier、Automation 或 worktree；把短任务硬做成 loop，只会增加流程负担。真正需要 loop 的任务，通常具备几个信号：会重复发生、要跨多轮推进、结果需要外部验收、会触碰风险边界，或者失败后需要留下证据给下一轮继续。

<div align=center><font size=2 color=#999999>任务形态判断表</font></div>
<div align="center">
<table width="80%">
<thead><tr><th>先问的问题</th><th>判断结果</th><th>优先走法</th></tr></thead>
<tbody>
<tr><td>任务能不能一轮完成，且结果容易人工看懂？</td><td>能</td><td>用 Prompt + 必要 Context，先不搭 loop。</td></tr>
<tr><td>任务是否每天、每周或按固定条件重复出现？</td><td>是</td><td>加入 Automation，再用 state 记录每次运行。</td></tr>
<tr><td>任务是否需要多轮修复，且完成条件可以写清？</td><td>是</td><td>使用 Goal；没有内置 Goal 时，用 state + verifier 手动推进。</td></tr>
<tr><td>任务是否会修改代码、文件、配置或外部系统？</td><td>是</td><td>加入 worktree、diff、权限边界和人工门。</td></tr>
<tr><td>任务结果是否难靠主 Agent 自己判断？</td><td>是</td><td>加入检测脚本、只读 verifier 或人工验收。</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;第二章的日报任务符合“定期重复 + 需要质量检查”的形态，所以 Automation、检测脚本、verifier 和 state 是必要部件。第三章的迁移任务符合“高风险修改 + 多轮修复 + 需要外部证据”的形态，所以 baseline、worktree、检查脚本、状态文件和两态 verifier 才是核心。这个判断做完，后面的部件选择就不会变成堆名词。

### 4.2 选最小部件组合

&emsp;&emsp;部件选择遵守一个原则：只补当前任务缺的能力。Prompt 已经能解决表达问题，就不要把它包装成流程；Context 已经能解决材料问题，就先别引入长期状态；只有当任务需要触发、记录、检查、修复和停止时，才逐层补 Automation、state、verifier、Goal、worktree 和人工门。

<div align=center><font size=2 color=#999999>最小部件组合表</font></div>
<div align="center">
<table width="80%">
<thead><tr><th>任务特征</th><th>最小组合</th><th>不要一开始就加</th></tr></thead>
<tbody>
<tr><td>一次性文字整理、局部解释、短内容改写。</td><td>Prompt + 少量 Context。</td><td>Automation、Goal、长期 state。</td></tr>
<tr><td>固定格式反复出现，例如 HTML 日报、审查清单、报告模板。</td><td>Skill + checklist + 输出检查。</td><td>复杂多 Agent 协作。</td></tr>
<tr><td>定期运行，例如每日 AI 新闻日报。</td><td>Automation + state + 检测脚本 + verifier。</td><td>worktree，除非任务会改代码。</td></tr>
<tr><td>多轮推进且完成条件明确。</td><td>Goal；没有内置 Goal 时用 state + verifier。</td><td>只靠聊天记录承接进度。</td></tr>
<tr><td>代码迁移、框架替换、跨文件修复。</td><td>baseline + worktree + 迁移地图 + 检查脚本 + diff + 人工门。</td><td>直接在主分支上连续改。</td></tr>
<tr><td>外部系统写入、删除、提交、发布。</td><td>connector / plugin / MCP + 权限规则 + 日志 + 人工门。</td><td>无人确认的自动写入。</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;这张表的价值在于克制。第二章没有引入 worktree，因为日报生成不是代码迁移；第三章没有只靠 Goal，因为迁移的关键不是“催着 Agent 继续”，而是让每轮都有文件级证据、运行检查和可审查 diff。部件越少，边界越清楚；边界清楚以后，再根据证据补下一层。

### 4.3 写首轮运行协议

&emsp;&emsp;决定开 loop 之后，第一轮先写运行协议，再让 Agent 执行。协议不是长 prompt，而是把目标、输入、边界、验收和记录位置放到外部，让下一轮、复核者和接手者都能读懂当前状态。第二章把这些信息放进日报任务、检测脚本和状态记录；第三章把它们落到迁移地图、检查脚本、状态文件、verifier 和项目规则里。

<div align=center><font size=2 color=#999999>首轮运行协议清单</font></div>
<div align="center">
<table width="80%">
<thead><tr><th>协议项</th><th>要写清什么</th><th>漏掉后的典型问题</th></tr></thead>
<tbody>
<tr><td>目标</td><td>这一轮要完成的可验收结果。</td><td>Agent 做了很多事，但没人知道是否完成。</td></tr>
<tr><td>输入材料</td><td>本轮可以读取哪些文件、网页、状态和前置产物。</td><td>材料不足时靠猜，或重复读错来源。</td></tr>
<tr><td>允许范围</td><td>可以修改哪些文件、生成哪些产物、调用哪些工具。</td><td>任务扩散，越改越远。</td></tr>
<tr><td>禁止动作</td><td>不得删除、提交、push、发布或写入外部系统等边界。</td><td>高风险动作先发生，人工只能事后补救。</td></tr>
<tr><td>验收方法</td><td>检测脚本、人工检查、verifier decision 或目标文件。</td><td>主 Agent 自己宣布完成，缺少外部证据。</td></tr>
<tr><td>状态记录</td><td>每轮目标、证据、检查结果、剩余问题写到哪里。</td><td>下一轮接不上，只能重新问一遍。</td></tr>
<tr><td>停止条件</td><td>complete、fix_required、rerun_required、blocked 分别怎么处理。</td><td>失败被无限重试，或可修问题被过早放弃。</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;这份协议最适合在任务开始前写成一页短文件。日报任务可以写成“检索范围、发布时间窗口、HTML 检查项、verifier 标准、state 更新位置”；迁移任务可以写成“baseline、worktree、迁移地图、禁止破坏的接口、静态检查、冒烟检查、真实任务样本、verifier 两态判断”。写清这些内容以后，loop 才有稳定的下一轮输入。

### 4.4 按根因修 loop

&emsp;&emsp;loop 跑偏时，不要只把问题归结为“Agent 不够好”。更有效的处理方式是回到结构里找根因：目标是否过大，材料是否不足，状态是否缺失，检查是否太弱，权限边界是否含糊，停止条件是否没写，沉淀位置是否混乱。根因不同，修法完全不同。

<div align=center><font size=2 color=#999999>跑偏归因表</font></div>
<div align="center">
<table width="80%">
<thead><tr><th>看到的现象</th><th>优先归因</th><th>下一轮修法</th></tr></thead>
<tbody>
<tr><td>prompt 已经很长，结果仍然偏。</td><td>目标、材料、检查和停止条件塞进了单轮输入。</td><td>拆成 Context、Harness、Loop 三层；把材料、检查和状态外部化。</td></tr>
<tr><td>定时任务能跑，但质量不稳定。</td><td>只有触发器，缺硬检查、verifier 和状态记录。</td><td>补检测脚本、只读复核者、state 和成功 / 失败分流。</td></tr>
<tr><td>Agent 声称完成，但证据不足。</td><td>执行者和验收者没有分离。</td><td>让 verifier 基于文件、diff、脚本输出给出判断。</td></tr>
<tr><td>迁移后服务能启动，但语义退化。</td><td>只检查运行状态，缺少真实任务样本。</td><td>补少量真实 case，检查关键输入输出是否保持一致。</td></tr>
<tr><td>同一问题反复修，轮次没有进展。</td><td>停止条件和无进展判断缺失。</td><td>设 max_rounds、连续无进展停止和人工确认点。</td></tr>
<tr><td>memory 越写越长，后续不好用。</td><td>本案例事实和可复用规则混在一起。</td><td>state 记录本轮事实，memory 只沉淀稳定经验。</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;归因完成后，下一轮修的不是“再试一次”，而是具体结构。材料不足就补 context pack；检查缺失就补脚本；验收不独立就加 verifier；风险边界不清就写项目协议；沉淀混乱就拆 state 和 memory。这样每一轮都比上一轮更清楚，而不是只把同一个任务重新交给 Agent。

### 4.5 停止条件与人工门

&emsp;&emsp;Loop Engineering 最容易被误解成“让 Agent 一直跑”。实际可用的 loop 一定可停止：完成时停止，缺事实时停止，越界时停止，连续无进展时停止，需要取舍时交给人。停止条件不是保守，它让自动化任务有明确责任边界。

<div align=center><font size=2 color=#999999>停止与分流表</font></div>
<div align="center">
<table width="80%">
<thead><tr><th>信号</th><th>判断</th><th>下一步</th></tr></thead>
<tbody>
<tr><td>检查通过，verifier 返回 complete / pass，state 已更新。</td><td>完成停止。</td><td>停止 loop，汇报产物、证据和剩余风险。</td></tr>
<tr><td>局部检查失败，但原因清楚、范围可控。</td><td>局部修复。</td><td>进入下一轮，只修失败项，不扩大目标。</td></tr>
<tr><td>来源、时间窗口、输入材料或前置判断失真。</td><td>前置重跑。</td><td>回到检索、读取、整理或规划阶段，不在错误材料上继续修。</td></tr>
<tr><td>缺原文、缺权限、缺业务判断或缺 API 凭证。</td><td>事实不足。</td><td>停止自动推进，请人补材料或确认取舍。</td></tr>
<tr><td>触碰删除、提交、push、merge、发布或外部系统写入。</td><td>高风险人工门。</td><td>展示意图、路径和 diff，等待确认。</td></tr>
<tr><td>连续多轮没有新证据，问题原样重复。</td><td>无进展停止。</td><td>汇报阻塞点，重新评估目标、上下文、权限或检查设计。</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;第二章的日报任务可以自动跑，但本地 HTML、检测结果、verifier decision 和状态记录必须同时成立才算完成。第三章的迁移任务可以让 Agent 多轮推进，但触碰关键 diff、项目边界、外部写入或架构取舍时要停下来。让 Agent 多做可验证的工作，让人只处理高风险判断，这是更稳的分工。

### 4.6 本章回顾

&emsp;&emsp;本章真正要带走的是一条工作顺序，而不是更多术语。先判断任务形态，决定是否值得进入 loop；再选择最小部件组合，避免把短任务做重；接着写首轮运行协议，让目标、材料、边界、验收和状态记录都在外部可见；运行中按根因修结构，最后用停止条件和人工门收束责任边界。

<div align=center><font size=2 color=#999999>Loop Engineering 一页带走清单</font></div>
<div align="center">
<table width="80%">
<thead><tr><th>顺序</th><th>要做的判断</th><th>一句话记法</th></tr></thead>
<tbody>
<tr><td>1</td><td>任务是否需要 loop。</td><td>短任务用 Prompt，长任务才外部化。</td></tr>
<tr><td>2</td><td>缺的是触发、复用、状态、检查、隔离还是权限。</td><td>缺什么补什么，不按工具名堆部件。</td></tr>
<tr><td>3</td><td>第一轮协议是否写清目标、材料、边界和验收。</td><td>先写清规则，再让 Agent 执行。</td></tr>
<tr><td>4</td><td>跑偏后归因到哪一层。</td><td>改 loop 结构，不只重试 prompt。</td></tr>
<tr><td>5</td><td>什么时候完成、重跑、停止或交给人。</td><td>可停止的循环才值得信任。</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;把全课压缩成一句话：Loop Engineering 不是把提示词写得更长，也不是多开几个 agent，而是把 AI 任务组织成<font color=red>可观察、可检查、可停止、可沉淀</font>的工程循环。第一章给我们四层责任边界，第二章给我们自动化日报的产品组合，第三章给我们手动迁移 loop 的工程资产；第四章把这些内容变成下次真实任务开工时可以直接使用的判断顺序。